# Counterfactual benchmark — four public forecasting datasets

Final public-dataset benchmark. Every non-target variable is editable.


In [ ]:
from __future__ import annotations

import json
import hashlib
import math
import os
import random
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from statsmodels.tsa.stattools import grangercausalitytests
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

# -----------------------------
# Reproducibility and utilities
# -----------------------------

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def now_stamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def format_float(x: float, ndigits: int = 5) -> str:
    if pd.isna(x):
        return "nan"
    return f"{float(x):.{ndigits}f}"


DEFAULT_DATASETS = ("ETTh1", "ETTh2", "Weather", "Electricity")


@dataclass
class EDBTConfig:
    # Paths. These mirror your local notebook structure. Adjust if needed.
    notebook_dir: Path = Path.cwd()
    data_root: Path = Path.cwd() / "data" / "Other"
    previous_output_root: Path = Path.cwd() / "outputs" / "EDBT_public_GRU_resolution_aware_v2"
    results_root: Path = Path.cwd() / "EDBT-results-public-datasets"

    # Dataset-specific input len, target and editable channels are assigned before running each dataset.
    datasets: Tuple[str, ...] = DEFAULT_DATASETS
    input_len: int = 168
    horizon: int = 24
    target_col: str = "OT"
    change_feature_names: Tuple[str, ...] = ()
    train_ratio: float = 0.5
    val_ratio: float = 0.1
    batch_size: int = 128
    num_workers: int = 0

    # GRU architecture of saved best models.
    gru_hidden: int = 100
    gru_layers: int = 2

    # Bound generation. COMET-style: trend from last target value toward a desired center, then trim to horizon.
    # Keep exactly the uploaded 2% bound formula for all domains.
    bound_reach_steps: int = 24
    desired_change_percent: float = -0.02  # -0.10
    bound_width_fraction: float = 0.50
    min_bound_width_std_fraction: float = 0.05

    # Evaluation. Set None for entire test set. Use a small number for debugging.
    # max_test_windows_per_dataset: Optional[int] = None
    max_test_windows_per_dataset: Optional[int] = 200
    eval_seed: int = 42
    resume_existing_results: bool = True

    # ForecastCF/COMET gradient search.
    cf_lr: float = 1e-2
    cf_max_iter: int = 100
    forecastcf_margin_weight: float = 1.0
    comet_margin_weight: float = 0.90
    comet_history_weight: float = 0.10
    comet_activity_threshold_original: float = 0.0
    comet_hist_round_decimals: int = 2
    comet_hist_tolerance_scaled: float = 1e-3

    # DiffusionCF-style local generative replacement.
    diffusion_window: int = 48
    diffusion_topk_anomalies: int = 5
    diffusion_candidates_per_window: int = 32
    vae_latent_dim: int = 8
    # vae_epochs: int = 20
    vae_epochs: int = 5
    vae_lr: float = 1e-3
    ddpm_steps: int = 100
    # ddpm_epochs: int = 20
    ddpm_epochs: int = 5
    vae_epochs: int = 5
    ddpm_lr: float = 1e-3

    # NTD-CFE adaptation.
    # ntd_max_episodes: int = 120
    ntd_max_episodes: int = 30
    ntd_max_steps_per_episode: int = 10
    ntd_lr: float = 1e-3
    ntd_gamma: float = 1.0
    ntd_step_scale: float = 0.15
    ntd_lambda_prox: float = 0.002
    ntd_validity_reward: bool = True

    # GA adaptation.
    ga_suffix_len: int = 48
    # ga_population: int = 60
    ga_population: int = 20
    # ga_generations: int = 35
    ga_generations: int = 12
    ga_mutation_prob: float = 0.25
    ga_crossover_prob: float = 0.75
    ga_tournament_size: int = 3
    ga_immigration_rate: float = 0.10
    ga_quantiles: Tuple[float, ...] = (0.1, 0.25, 0.5, 0.75, 0.9)
    ga_lag_order: int = 6
    ga_granger_alpha: float = 0.05
    ga_likelihood_weight: float = 0.02
    ga_proximity_weight: float = 0.05

    # General.
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = EDBTConfig()
set_seed(cfg.seed)
print("Device:", cfg.device)
print("Default maximum test windows per dataset:", cfg.max_test_windows_per_dataset)

## Environment and saved-model selection

In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

import os, sys, platform


**Change MODEL_RUNS after getting the best models**

In [ ]:
# Exact CSV names and paths reflect your latest uploaded GRU_4_public_datasets_1.ipynb.
DATA_FILES = {
    'ETTh1': PROJECT_ROOT/'Data'/'Other'/'ETT-small'/'ETTh1.csv',
    'ETTh2': PROJECT_ROOT/'Data'/'Other'/'ETT-small'/'ETTh2.csv',
    'Weather': PROJECT_ROOT/'Data'/'Other'/'weather'/'weather_cleaned.csv',
    'Electricity': PROJECT_ROOT/'Data'/'Other'/'electricity'/'electricity.csv',
}
# Use the validation-selected saved forecasters produced by notebook 03.
PUBLIC_MODEL_ROOT = PROJECT_ROOT / 'Results' / 'EDBT_public_GRU_v6'
TARGETS = {'ETTh1':'OT', 'ETTh2':'OT', 'Weather':'T (degC)', 'Electricity':'319'}
EXPECTED_FEATURE_COUNTS = {'ETTh1':7, 'ETTh2':7, 'Weather':21, 'Electricity':321}
EXPECTED_OBSERVATIONS = {'ETTh1':17420, 'ETTh2':17420, 'Weather':52704, 'Electricity':26304}
EXPECTED_WINDOWS = {'ETTh1':(336,24), 'ETTh2':(336,24), 'Weather':(288,24), 'Electricity':(168,24)}
# Final intervention space: all non-target input variables are editable.
# New sibling of EDBT-results. No writes occur inside old BRGM results or public model folders.
RESULTS_ROOT_PUBLIC = PROJECT_ROOT/'Results'/'EDBT-results-public-datasets'
# For a 2-window test use a DIFFERENT path, e.g. EDBT-results-public-PILOT.
# The source defaults to 200 test windows and seven methods; set cfg.max_test_windows_per_dataset=2 for a pilot.
cfg.results_root = RESULTS_ROOT_PUBLIC
cfg.notebook_dir = PROJECT_ROOT
cfg.data_root = PROJECT_ROOT/'Data'/'Other'
cfg.bound_reach_steps = 24
cfg.save_cf_example_plots = True
cfg.example_plot_every = 20  # the source's implicit default
cfg.example_plot_methods = ["ForecastCF-target", "ForecastCF-exog", "COMET-exog", "DiffusionCF-target", "DiffusionCF-exog", "NTD-CFE-exog", "GA-exog"]
cfg.example_plot_max_per_dataset = None
print('New CF result root:',cfg.results_root)

ACTIVE_DATASET = None
ACTIVE_MODEL_ROOT = None
ACTIVE_MODEL_METADATA = None

def resolve_csv_path(name):
    p = Path(DATA_FILES[name]).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(f'{name}: missing CSV at {p}. Change DATA_FILES[{name!r}].')
    if name == 'Weather' and p.name != 'weather_cleaned.csv':
        raise ValueError('Weather MUST use weather_cleaned.csv (52,704 rows), not weather.csv')
    return p

def select_editable_features(name, csv_path):

    h = pd.read_csv(
        csv_path,
        nrows=0
    ).columns.tolist()

    feats = [
        c for c in h
        if c != 'date'
    ]

    target = TARGETS[name]

    chosen = tuple(
        f for f in feats
        if f != target
    )

    return chosen

def configure_public_dataset(name):
    global ACTIVE_DATASET, ACTIVE_MODEL_ROOT, ACTIVE_MODEL_METADATA
    if name not in DATA_FILES: raise KeyError(name)
    if not RESULTS_ROOT_PUBLIC.is_absolute(): raise ValueError('Results root must be absolute')
    csv_path = resolve_csv_path(name)
    run_root = PUBLIC_MODEL_ROOT
    meta_path=run_root/'datasets'/name/'best_gru_metadata.json'
    if not meta_path.exists():
        raise FileNotFoundError(f'Missing model metadata {meta_path}; run notebook 03 first.')
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    actual_hash=sha256_file(csv_path)
    if meta.get('csv_sha256') != actual_hash:
        raise ValueError(f'{name}: selected CSV SHA256 differs from model training CSV. Check DATA_FILES/version; do not force-run.')
    tr=meta.get('configuration', {})
    expected_l,expected_h=EXPECTED_WINDOWS[name]
    if tr.get('input_len') != expected_l or tr.get('horizon') != expected_h:
        raise ValueError(f'{name}: checkpoint metadata reports history/horizon {tr.get("input_len")}/{tr.get("horizon")}, expected {expected_l}/{expected_h}. Update configuration only deliberately.')
    if meta.get('target_col') != TARGETS[name]:
        raise ValueError(f'{name}: wrong model target {meta.get("target_col")}')
    if float(tr.get('train_ratio',-1)) != cfg.train_ratio or float(tr.get('val_ratio',-1)) != cfg.val_ratio:
        raise ValueError(f'{name}: training split incompatible with source benchmark')
    if tr.get('gru_hidden') != cfg.gru_hidden or tr.get('gru_layers') != cfg.gru_layers:
        raise ValueError(f'{name}: loaded GRU architecture differs')
    chosen=select_editable_features(name,csv_path)
    ACTIVE_DATASET=name; ACTIVE_MODEL_ROOT=run_root; ACTIVE_MODEL_METADATA=meta
    cfg.datasets=(name,)
    cfg.previous_output_root=run_root
    cfg.target_col=TARGETS[name]
    cfg.change_feature_names=chosen
    cfg.input_len=tr['input_len'];cfg.horizon=tr['horizon']
    # Keep all hyperparameters and evaluation metrics untouched, including GA/diffusion 48-step suffix/window.
    cfg.bound_reach_steps=24
    set_seed(cfg.seed)
    print(f'\n{name}: CSV={csv_path}\nmodel={run_root}\nfeatures={len(meta["feature_cols"])}; target={cfg.target_col}; editable={cfg.change_feature_names}; input={cfg.input_len}; horizon={cfg.horizon}; dropout (training only)={tr.get("gru_dropout",0.0)}')
    return csv_path

## Original window loader / GRU architecture; dataset-specific integrity checks

In [ ]:
# -----------------------------
# Data preparation and GRU loading
# -----------------------------

class SlidingWindowDataset(Dataset):
    def __init__(self, full_scaled: np.ndarray, start: int, end: int, input_len: int, horizon: int, target_idx: int) -> None:
        self.full_scaled = full_scaled.astype(np.float32)
        self.start = int(start)
        self.end = int(end)
        self.input_len = int(input_len)
        self.horizon = int(horizon)
        self.target_idx = int(target_idx)
        self.segment_len = max(self.end - self.start, 0)
        self.num_samples = max(self.segment_len - self.input_len - self.horizon, 0)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        start_idx = self.start + int(idx)
        x = self.full_scaled[start_idx : start_idx + self.input_len]
        y = self.full_scaled[start_idx + self.input_len : start_idx + self.input_len + self.horizon, self.target_idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), start_idx


@dataclass
class PreparedDataset:
    name: str
    csv_path: Path
    df: pd.DataFrame
    feature_cols: List[str]
    target_idx: int
    change_feature_idx: List[int]
    static_feature_idx: List[int]
    scaler: StandardScaler
    values_original: np.ndarray
    values_scaled: np.ndarray
    train_range: Tuple[int, int]
    val_range: Tuple[int, int]
    test_range: Tuple[int, int]
    train_ds: SlidingWindowDataset
    val_ds: SlidingWindowDataset
    test_ds: SlidingWindowDataset

    @property
    def n_features(self) -> int:
        return len(self.feature_cols)

    @property
    def train_original(self) -> np.ndarray:
        return self.values_original[self.train_range[0] : self.train_range[1]]

    @property
    def train_scaled(self) -> np.ndarray:
        return self.values_scaled[self.train_range[0] : self.train_range[1]]

    def inverse_window(self, x_scaled: np.ndarray) -> np.ndarray:
        return self.scaler.inverse_transform(x_scaled)

    def scale_window(self, x_original: np.ndarray) -> np.ndarray:
        return self.scaler.transform(x_original).astype(np.float32)

    def inverse_target(self, y_scaled: np.ndarray) -> np.ndarray:
        return y_scaled * self.scaler.scale_[self.target_idx] + self.scaler.mean_[self.target_idx]

    def scale_target(self, y_original: np.ndarray) -> np.ndarray:
        return ((y_original - self.scaler.mean_[self.target_idx]) / self.scaler.scale_[self.target_idx]).astype(np.float32)

    def make_loader(self, split: str, shuffle: bool = False) -> DataLoader:
        ds = {"train": self.train_ds, "val": self.val_ds, "test": self.test_ds}[split]
        return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle, num_workers=cfg.num_workers)


class GRUForecaster(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, horizon: int):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=0.0)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])



def discover_dataset_files(data_root: Path, datasets: Sequence[str]) -> Tuple[List[Path], List[str]]:
    # Exact, fail-closed paths from configuration, not first matching recursive CSV.
    paths, missing = [], []
    for name in datasets:
        try: paths.append(resolve_csv_path(name))
        except FileNotFoundError: missing.append(name)
    return paths, missing


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(4*1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_piezo_dataframe(csv_path: Path) -> pd.DataFrame:
    # Named as in original notebook; now handles each public dataset's original CSV.
    df = pd.read_csv(csv_path)
    required = {'date', cfg.target_col, *cfg.change_feature_names}
    missing = sorted(required - set(df.columns))
    if missing: raise ValueError(f'{csv_path.name}: missing required CSV columns {missing}')
    df['date'] = pd.to_datetime(df['date'], errors='raise')
    if df['date'].isna().any() or df['date'].duplicated().any():
        raise ValueError(f'{csv_path.name}: invalid or duplicated dates, check cleaned Weather CSV')
    df = df.sort_values('date').reset_index(drop=True)
    feat = [c for c in df.columns if c != 'date']
    if len(feat) != EXPECTED_FEATURE_COUNTS[ACTIVE_DATASET]:
        raise ValueError(f'{csv_path.name}: expected {EXPECTED_FEATURE_COUNTS[ACTIVE_DATASET]} numeric channels; got {len(feat)}')
    if len(df) != EXPECTED_OBSERVATIONS[ACTIVE_DATASET]:
        raise ValueError(f'{csv_path.name}: row count mismatch: expected {EXPECTED_OBSERVATIONS[ACTIVE_DATASET]}, got {len(df)}')
    vals = df[feat].to_numpy(dtype=np.float32)
    if not np.isfinite(vals).all():
        raise ValueError(f'{csv_path.name}: NaN/inf values. Do NOT silently impute: trained model expects matching CSV.')
    return df


def prepare_dataset(csv_path: Path) -> PreparedDataset:
    # The next 42 lines reproduce the original prepare_dataset (scaler, windows, splits).
    df = load_piezo_dataframe(csv_path)
    feature_cols = [c for c in df.columns if c != 'date']
    target_idx = feature_cols.index(cfg.target_col)
    change_feature_idx = [feature_cols.index(c) for c in cfg.change_feature_names]
    static_feature_idx = [i for i in range(len(feature_cols)) if i not in change_feature_idx]
    values = df[feature_cols].values.astype(np.float32)
    n = len(values)
    n_train = int(n * cfg.train_ratio)
    n_val = int(n * cfg.val_ratio)
    scaler = StandardScaler()
    scaler.fit(values[:n_train])
    scaled = scaler.transform(values).astype(np.float32)
    train_range = (0, n_train)
    val_range = (n_train - cfg.input_len - cfg.horizon, n_train + n_val)
    test_range = (n_train + n_val - cfg.input_len - cfg.horizon, n)
    train_ds = SlidingWindowDataset(scaled, *train_range, cfg.input_len, cfg.horizon, target_idx)
    val_ds = SlidingWindowDataset(scaled, *val_range, cfg.input_len, cfg.horizon, target_idx)
    test_ds = SlidingWindowDataset(scaled, *test_range, cfg.input_len, cfg.horizon, target_idx)
    if min(len(train_ds), len(val_ds), len(test_ds)) <= 0:
        raise ValueError(f'{csv_path.name} empty split with input_len={cfg.input_len}, horizon={cfg.horizon}')
    bundle = PreparedDataset(ACTIVE_DATASET, csv_path, df, feature_cols, target_idx, change_feature_idx,
                             static_feature_idx, scaler, values, scaled, train_range, val_range, test_range,
                             train_ds, val_ds, test_ds)
    # Assert against training-time scaler, feature ordering and chronological split.
    meta = ACTIVE_MODEL_METADATA
    if feature_cols != meta['feature_cols']:
        raise ValueError('Feature order differs from training checkpoint metadata')
    if list(train_range) != list(meta['train_range']) or list(val_range) != list(meta['val_range']) or list(test_range) != list(meta['test_range']):
        raise ValueError('Chronological split differs from checkpoint training metadata')
    sp = ACTIVE_MODEL_ROOT / 'datasets' / ACTIVE_DATASET / 'scaler_parameters.npz'
    if not sp.exists(): raise FileNotFoundError(f'Missing training-time scaler: {sp}')
    old = np.load(sp)
    if not np.allclose(scaler.mean_, old['mean'], rtol=1e-6, atol=1e-6): raise ValueError('Scaler mean mismatch')
    if not np.allclose(scaler.scale_, old['scale'], rtol=1e-6, atol=1e-6): raise ValueError('Scaler scale mismatch')
    return bundle


def find_saved_model_path(dataset_name: str) -> Path:
    p = cfg.previous_output_root / 'datasets' / dataset_name / 'best_gru_forecaster.pth'
    if not p.is_file(): raise FileNotFoundError(f'Explicit checkpoint not found: {p}')
    return p


def load_best_gru(bundle: PreparedDataset) -> nn.Module:
    path = find_saved_model_path(bundle.name)
    try: state = torch.load(path, map_location=cfg.device, weights_only=True)
    except TypeError: state = torch.load(path, map_location=cfg.device)
    if not isinstance(state, dict) or 'gru.weight_ih_l0' not in state or 'fc.weight' not in state:
        raise ValueError('Expected bare GRU state dict; check selected checkpoint')
    hidden = state['gru.weight_ih_l0'].shape[0] // 3
    input_size = state['gru.weight_ih_l0'].shape[1]
    horizon = state['fc.weight'].shape[0]
    if input_size != bundle.n_features or horizon != cfg.horizon or hidden != cfg.gru_hidden:
        raise ValueError(f'GRU checkpoint shape mismatch: input={input_size}, horizon={horizon}, hidden={hidden}; configured {bundle.n_features}, {cfg.horizon}, {cfg.gru_hidden}')
    # Train-time dropout has no weights and is disabled at prediction time. Instantiate
    # dropout=0 so original GradientCF.train() cannot make this fixed forecaster stochastic.
    model = GRUForecaster(input_size=input_size, hidden_size=hidden, num_layers=cfg.gru_layers, horizon=horizon).to(cfg.device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


@torch.no_grad()
def predict_model(model: nn.Module, x_np: np.ndarray) -> np.ndarray:
    model.eval()
    x = torch.tensor(x_np[None, ...], dtype=torch.float32, device=cfg.device)
    return model(x).detach().cpu().numpy()[0]


@torch.no_grad()
def evaluate_forecaster(model: nn.Module, bundle: PreparedDataset) -> Dict[str, float]:
    model.eval()
    ys, ps = [], []
    for xb, yb, _ in bundle.make_loader("test", shuffle=False):
        xb = xb.to(cfg.device)
        pred = model(xb).detach().cpu().numpy()
        ys.append(yb.numpy())
        ps.append(pred)
    y = np.concatenate(ys, axis=0)
    p = np.concatenate(ps, axis=0)
    y_orig = np.stack([bundle.inverse_target(row) for row in y])
    p_orig = np.stack([bundle.inverse_target(row) for row in p])
    return {
        "test_mae": float(mean_absolute_error(y_orig.ravel(), p_orig.ravel())),
        "test_rmse": float(math.sqrt(mean_squared_error(y_orig.ravel(), p_orig.ravel()))),
        "test_r2": float(r2_score(y_orig.ravel(), p_orig.ravel())),
        "n_test_windows": int(len(y_orig)),
    }

## Original bounds, evaluation metrics, and method implementations (unchanged)

In [ ]:
# -----------------------------
# COMET-style desired bounds and shared metrics
# -----------------------------

def generate_comet_style_bounds(bundle: PreparedDataset, x_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return upper/lower in scaled model units and in original target units.

    COMET generates bounds by following a trend toward a desired center and trimming the vector to the forecast horizon.
    In glucose, the desired center is a clinical threshold. For piezometers, we use a configurable relative change from
    the last historical level as the desired center.
    """
    x_orig = bundle.inverse_window(x_scaled)
    target_hist = x_orig[:, bundle.target_idx]
    last_val = float(target_hist[-1])
    std_val = float(np.std(target_hist))
    width = max(cfg.bound_width_fraction * std_val, cfg.min_bound_width_std_fraction * std_val, 1e-6)
    desired_center = last_val * (1.0 + cfg.desired_change_percent)
    reach_steps = max(cfg.bound_reach_steps, cfg.horizon)
    center_path = np.linspace(last_val, desired_center, reach_steps, dtype=np.float32)[: cfg.horizon]
    lower_orig = center_path - width
    upper_orig = center_path + width
    lower_scaled = bundle.scale_target(lower_orig)
    upper_scaled = bundle.scale_target(upper_orig)
    return upper_scaled.astype(np.float32), lower_scaled.astype(np.float32), upper_orig.astype(np.float32), lower_orig.astype(np.float32)


def validity_mask(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> np.ndarray:
    return (pred_orig <= upper_orig) & (pred_orig >= lower_orig)


def per_step_validity_dict(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray,
                           prefix: str = "valid_h") -> Dict[str, int]:
    """Return one binary validity column per forecast horizon step.

    Example:
        valid_h_01 = 1 if prediction at horizon step 1 is inside [lower, upper], otherwise 0.
    """
    mask = validity_mask(
        np.asarray(pred_orig).reshape(-1),
        np.asarray(upper_orig).reshape(-1),
        np.asarray(lower_orig).reshape(-1),
    ).astype(int)

    return {
        f"{prefix}_{i + 1:02d}": int(mask[i])
        for i in range(len(mask))
    }


def validity_ratio_metric(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> float:
    return float(np.mean(validity_mask(pred_orig, upper_orig, lower_orig)))


def prefix_consecutive_valid_steps(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> int:
    mask = validity_mask(pred_orig, upper_orig, lower_orig)
    count = 0
    for v in mask:
        if bool(v):
            count += 1
        else:
            break
    return int(count)


def max_consecutive_valid_steps(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> int:
    mask = validity_mask(pred_orig, upper_orig, lower_orig)
    best = cur = 0
    for v in mask:
        if bool(v):
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return int(best)


def stepwise_validity_auc_metric(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> float:
    # ForecastCF-style cumulative validity from the beginning of the horizon.
    T = len(pred_orig)
    if T == 0:
        return 0.0
    prefix = prefix_consecutive_valid_steps(pred_orig, upper_orig, lower_orig)
    return float(prefix / T)


def distance_to_band(pred_orig: np.ndarray, upper_orig: np.ndarray, lower_orig: np.ndarray) -> float:
    below = np.maximum(lower_orig - pred_orig, 0.0)
    above = np.maximum(pred_orig - upper_orig, 0.0)
    return float(np.mean(below + above))


def proximity_metric(x_orig_scaled: np.ndarray, x_cf_scaled: np.ndarray) -> float:
    diff = x_cf_scaled - x_orig_scaled
    return float(np.linalg.norm(diff) / math.sqrt(diff.size))


def l1_proximity_metric(x_orig_scaled: np.ndarray, x_cf_scaled: np.ndarray) -> float:
    return float(np.mean(np.abs(x_cf_scaled - x_orig_scaled)))


def sparsity_count_metric(x_orig_scaled: np.ndarray, x_cf_scaled: np.ndarray, tol: float = 1e-4) -> int:
    return int(np.sum(np.abs(x_cf_scaled - x_orig_scaled) > tol))


def sparsity_ratio_metric(x_orig_scaled: np.ndarray, x_cf_scaled: np.ndarray, tol: float = 1e-4) -> float:
    return float(np.mean(np.abs(x_cf_scaled - x_orig_scaled) > tol))


def evaluate_cf(bundle: PreparedDataset, x_orig_scaled: np.ndarray, x_cf_scaled: np.ndarray, pred_cf_scaled: np.ndarray,
                upper_orig: np.ndarray, lower_orig: np.ndarray, runtime_sec: float) -> Dict[str, float]:
    pred_cf_orig = bundle.inverse_target(pred_cf_scaled)
    return {
        "validity_ratio": validity_ratio_metric(pred_cf_orig, upper_orig, lower_orig),
        "stepwise_validity_auc": stepwise_validity_auc_metric(pred_cf_orig, upper_orig, lower_orig),
        "consecutive_valid_steps": prefix_consecutive_valid_steps(pred_cf_orig, upper_orig, lower_orig),
        "max_consecutive_valid_steps": max_consecutive_valid_steps(pred_cf_orig, upper_orig, lower_orig),
        "distance_to_band": distance_to_band(pred_cf_orig, upper_orig, lower_orig),
        "proximity_l2_scaled": proximity_metric(x_orig_scaled, x_cf_scaled),
        "proximity_l1_scaled": l1_proximity_metric(x_orig_scaled, x_cf_scaled),
        "sparsity_count": sparsity_count_metric(x_orig_scaled, x_cf_scaled),
        "sparsity_ratio": sparsity_ratio_metric(x_orig_scaled, x_cf_scaled),
        "runtime_sec": float(runtime_sec),
    }


def interval_loss_torch(pred: torch.Tensor, upper: torch.Tensor, lower: torch.Tensor) -> torch.Tensor:
    # Clean interval loss: zero inside bounds, squared violation outside.
    below = torch.clamp(lower - pred, min=0.0)
    above = torch.clamp(pred - upper, min=0.0)
    return torch.sum(below ** 2 + above ** 2)

In [ ]:
# -----------------------------
# ForecastCF and COMET gradient methods
# -----------------------------

class GradientCF:
    def __init__(self, model: nn.Module, change_idx: Sequence[int], lr: float, max_iter: int, margin_weight: float,
                 clip_ranges: Optional[Dict[int, Tuple[float, float]]] = None,
                 activity_weights: Optional[np.ndarray] = None,
                 historical_values: Optional[Dict[int, np.ndarray]] = None,
                 history_weight: float = 0.0,
                 hist_tolerance: float = 1e-3,
                 name: str = "GradientCF") -> None:
        self.model = model
        self.change_idx = list(change_idx)
        self.lr = lr
        self.max_iter = max_iter
        self.margin_weight = margin_weight
        self.prox_weight = max(0.0, 1.0 - margin_weight)
        self.clip_ranges = clip_ranges or {}
        self.activity_weights = activity_weights
        self.historical_values = historical_values or {}
        self.history_weight = history_weight
        self.hist_tolerance = hist_tolerance
        self.name = name

    def _weighted_ape(self, x0: torch.Tensor, xcf: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
        denom = torch.clamp(torch.abs(x0), min=1e-6)
        return torch.mean(torch.abs(x0 - xcf) / denom * weights)

    def _historical_penalty(self, xcf: torch.Tensor) -> torch.Tensor:
        if not self.historical_values:
            return torch.tensor(0.0, dtype=torch.float32, device=cfg.device)
        pen = torch.tensor(0.0, dtype=torch.float32, device=cfg.device)
        for feat in self.change_idx:
            vals = xcf[0, :, feat].unsqueeze(1)
            hist = torch.tensor(self.historical_values[feat], dtype=torch.float32, device=cfg.device).unsqueeze(0)
            near = torch.min(torch.abs(vals - hist), dim=1).values
            pen = pen + torch.mean(torch.clamp(near - self.hist_tolerance, min=0.0))
        return pen

    def generate(self, x0_np: np.ndarray, upper_scaled: np.ndarray, lower_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
        start = time.time()
        # cuDNN RNN backward can require train mode; saved GRU has no dropout, and parameters are frozen.
        self.model.train()
        for p in self.model.parameters():
            p.requires_grad_(False)

        x0 = torch.tensor(x0_np[None, ...], dtype=torch.float32, device=cfg.device)
        xcf = x0.clone().detach().requires_grad_(True)
        upper = torch.tensor(upper_scaled, dtype=torch.float32, device=cfg.device)
        lower = torch.tensor(lower_scaled, dtype=torch.float32, device=cfg.device)
        opt = torch.optim.Adam([xcf], lr=self.lr)

        fixed_mask = torch.ones_like(x0)
        fixed_mask[:, :, self.change_idx] = 0.0
        free_mask = 1.0 - fixed_mask

        if self.activity_weights is None:
            weights = torch.ones_like(x0)
        else:
            weights = torch.tensor(self.activity_weights[None, ...], dtype=torch.float32, device=cfg.device)

        final_loss = float("nan")
        for _ in range(self.max_iter):
            pred = self.model(xcf).squeeze(0)
            margin = interval_loss_torch(pred, upper, lower)
            prox = self._weighted_ape(x0, xcf, weights)
            hist = self._historical_penalty(xcf)
            loss = self.margin_weight * margin + self.prox_weight * prox + self.history_weight * hist
            opt.zero_grad()
            loss.backward()
            if xcf.grad is not None:
                xcf.grad *= free_mask
            opt.step()
            with torch.no_grad():
                xcf.data = fixed_mask * x0 + free_mask * xcf.data
                for feat, (lo, hi) in self.clip_ranges.items():
                    if feat in self.change_idx:
                        xcf.data[:, :, feat].clamp_(float(lo), float(hi))
                pred_after = self.model(xcf).squeeze(0)
            final_loss = float(loss.detach().cpu().item())
            if bool(torch.all((pred_after <= upper) & (pred_after >= lower))):
                break

        with torch.no_grad():
            cf = xcf.detach().cpu().numpy()[0].astype(np.float32)
            pred_cf = self.model(torch.tensor(cf[None, ...], dtype=torch.float32, device=cfg.device)).detach().cpu().numpy()[0]
        return cf, pred_cf.astype(np.float32), time.time() - start


def build_clip_ranges(bundle: PreparedDataset, feature_indices: Sequence[int]) -> Dict[int, Tuple[float, float]]:
    ranges = {}
    train = bundle.train_scaled
    for feat in feature_indices:
        ranges[feat] = (float(np.min(train[:, feat])), float(np.max(train[:, feat])))
    return ranges


def build_historical_values(bundle: PreparedDataset, feature_indices: Sequence[int]) -> Dict[int, np.ndarray]:
    hist = {}
    train = bundle.train_scaled
    for feat in feature_indices:
        hist[feat] = np.unique(np.round(train[:, feat], cfg.comet_hist_round_decimals)).astype(np.float32)
    return hist


def build_activity_weights(bundle: PreparedDataset, x_scaled: np.ndarray, feature_indices: Sequence[int]) -> np.ndarray:
    # COMET uses C=0 to encourage changes at planned activity times and C=1 otherwise.
    # For piezometer covariates, only precipitation has a clear inactive zero state. Temperature/ETP are always active.
    weights = np.ones_like(x_scaled, dtype=np.float32)
    x_orig = bundle.inverse_window(x_scaled)
    for feat in feature_indices:
        name = bundle.feature_cols[feat].lower()
        if any(key in name for key in ["preliq", "rain", "precip"]):
            active = x_orig[:, feat] > cfg.comet_activity_threshold_original
            weights[:, feat] = np.where(active, 0.0, 1.0)
        else:
            # For the public datasets make all the exogenous variables active
            weights[:, feat] = 1.0
    return weights


def make_forecastcf(model: nn.Module, bundle: PreparedDataset, change_idx: Sequence[int], name: str) -> GradientCF:
    return GradientCF(model, change_idx=change_idx, lr=cfg.cf_lr, max_iter=cfg.cf_max_iter,
                      margin_weight=cfg.forecastcf_margin_weight, name=name)


def make_comet(model: nn.Module, bundle: PreparedDataset, x_scaled: np.ndarray) -> GradientCF:
    change_idx = bundle.change_feature_idx
    return GradientCF(
        model, change_idx=change_idx, lr=cfg.cf_lr, max_iter=cfg.cf_max_iter,
        margin_weight=cfg.comet_margin_weight,
        clip_ranges=build_clip_ranges(bundle, change_idx),
        activity_weights=build_activity_weights(bundle, x_scaled, change_idx),
        historical_values=build_historical_values(bundle, change_idx),
        history_weight=cfg.comet_history_weight,
        hist_tolerance=cfg.comet_hist_tolerance_scaled,
        name="COMET-exog",
    )

In [ ]:
# -----------------------------
# DiffusionCF-style VAE + DDPM window replacement
# -----------------------------

class ChangeWindowDataset(Dataset):
    def __init__(self, base_ds: SlidingWindowDataset, change_idx: Sequence[int], window_size: int):
        self.windows = []
        self.change_idx = list(change_idx)
        self.window_size = int(window_size)
        for i in range(len(base_ds)):
            x, _, _ = base_ds[i]
            arr = x.numpy()[:, self.change_idx]
            if len(arr) >= self.window_size:
                for start in range(0, len(arr) - self.window_size + 1, max(1, self.window_size // 2)):
                    self.windows.append(arr[start:start+self.window_size].reshape(-1).astype(np.float32))
        if len(self.windows) == 0:
            self.windows = [np.zeros(self.window_size * len(self.change_idx), dtype=np.float32)]

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return torch.tensor(self.windows[idx], dtype=torch.float32)


class WindowVAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        hidden = max(64, min(256, input_dim * 2))
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, input_dim))

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        rec = self.decoder(z)
        return rec, mu, logvar

    @torch.no_grad()
    def reconstruction_error(self, x_np: np.ndarray) -> float:
        self.eval()
        x = torch.tensor(x_np[None, :], dtype=torch.float32, device=cfg.device)
        rec, _, _ = self.forward(x)
        return float(torch.mean((rec - x) ** 2).detach().cpu().item())


def vae_loss(rec, x, mu, logvar):
    recon = F.mse_loss(rec, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + 1e-3 * kl


class TimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    def forward(self, t: torch.Tensor):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device) * -(math.log(10000.0) / max(half - 1, 1)))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class EpsModel(nn.Module):
    def __init__(self, dim: int, tdim: int = 32):
        super().__init__()
        hidden = max(128, min(512, dim * 2))
        self.temb = TimeEmbedding(tdim)
        self.net = nn.Sequential(nn.Linear(dim + tdim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, dim))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=1))


class TinyDDPM(nn.Module):
    def __init__(self, dim: int, steps: int):
        super().__init__()
        self.dim = dim
        self.steps = steps
        self.eps_model = EpsModel(dim)
        beta = torch.linspace(1e-4, 0.02, steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        self.register_buffer("beta", beta)
        self.register_buffer("alpha", alpha)
        self.register_buffer("alpha_bar", alpha_bar)

    def loss(self, x0):
        b = x0.shape[0]
        t = torch.randint(0, self.steps, (b,), device=x0.device)
        noise = torch.randn_like(x0)
        ab = self.alpha_bar[t].unsqueeze(1)
        xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
        pred = self.eps_model(xt, t)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def sample(self, n: int):
        self.eval()
        x = torch.randn(n, self.dim, device=cfg.device)
        for t_int in reversed(range(self.steps)):
            t = torch.full((n,), t_int, device=cfg.device, dtype=torch.long)
            eps = self.eps_model(x, t)
            alpha_t = self.alpha[t_int]
            ab_t = self.alpha_bar[t_int]
            beta_t = self.beta[t_int]
            mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - ab_t)) * eps)
            if t_int > 0:
                x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
            else:
                x = mean
        return x.detach().cpu().numpy().astype(np.float32)


def train_vae_for(bundle: PreparedDataset, change_idx: Sequence[int]) -> WindowVAE:
    ds = ChangeWindowDataset(bundle.train_ds, change_idx, cfg.diffusion_window)
    loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=cfg.num_workers)
    dim = cfg.diffusion_window * len(change_idx)
    vae = WindowVAE(dim, cfg.vae_latent_dim).to(cfg.device)
    opt = torch.optim.Adam(vae.parameters(), lr=cfg.vae_lr)
    vae.train()
    for _ in range(cfg.vae_epochs):
        for xb in loader:
            xb = xb.to(cfg.device)
            rec, mu, logvar = vae(xb)
            loss = vae_loss(rec, xb, mu, logvar)
            opt.zero_grad(); loss.backward(); opt.step()
    return vae


def train_ddpm_for(bundle: PreparedDataset, change_idx: Sequence[int]) -> TinyDDPM:
    ds = ChangeWindowDataset(bundle.train_ds, change_idx, cfg.diffusion_window)
    loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=cfg.num_workers)
    dim = cfg.diffusion_window * len(change_idx)
    ddpm = TinyDDPM(dim, cfg.ddpm_steps).to(cfg.device)
    opt = torch.optim.Adam(ddpm.parameters(), lr=cfg.ddpm_lr)
    ddpm.train()
    for _ in range(cfg.ddpm_epochs):
        for xb in loader:
            xb = xb.to(cfg.device)
            loss = ddpm.loss(xb)
            opt.zero_grad(); loss.backward(); opt.step()
    return ddpm


class DiffusionCFAdapter:
    def __init__(self, model: nn.Module, change_idx: Sequence[int], vae: WindowVAE, ddpm: TinyDDPM, train_scaled: np.ndarray, name: str):
        self.model = model
        self.change_idx = list(change_idx)
        self.vae = vae
        self.ddpm = ddpm
        self.train_scaled = train_scaled
        self.name = name

    def _rank_windows(self, x_scaled: np.ndarray) -> List[Tuple[float, int]]:
        local = x_scaled[:, self.change_idx]
        out = []
        for start in range(0, len(local) - cfg.diffusion_window + 1):
            w = local[start:start+cfg.diffusion_window].reshape(-1).astype(np.float32)
            out.append((self.vae.reconstruction_error(w), start))
        return sorted(out, reverse=True)[: cfg.diffusion_topk_anomalies]

    def generate(self, x0_np: np.ndarray, upper_scaled: np.ndarray, lower_scaled: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
        start_time = time.time()
        best_x = x0_np.copy()
        best_pred = predict_model(self.model, best_x)
        best_score = float("inf")
        upper_t = torch.tensor(upper_scaled, dtype=torch.float32)
        lower_t = torch.tensor(lower_scaled, dtype=torch.float32)
        feat_min = self.train_scaled[:, self.change_idx].min(axis=0)
        feat_max = self.train_scaled[:, self.change_idx].max(axis=0)
        for _, start in self._rank_windows(x0_np):
            candidates = self.ddpm.sample(cfg.diffusion_candidates_per_window).reshape(cfg.diffusion_candidates_per_window, cfg.diffusion_window, len(self.change_idx))
            candidates = np.clip(candidates, feat_min[None, None, :], feat_max[None, None, :])
            for cand in candidates:
                xcf = x0_np.copy()
                xcf[start:start+cfg.diffusion_window, self.change_idx] = cand
                pred = predict_model(self.model, xcf)
                pred_t = torch.tensor(pred, dtype=torch.float32)
                margin = float(interval_loss_torch(pred_t, upper_t, lower_t).item())
                prox = proximity_metric(x0_np, xcf)
                score = margin + 0.05 * prox
                if score < best_score:
                    best_score = score
                    best_x = xcf.astype(np.float32)
                    best_pred = pred.astype(np.float32)
                if np.all((pred <= upper_scaled) & (pred >= lower_scaled)):
                    return best_x, best_pred, time.time() - start_time
        return best_x, best_pred, time.time() - start_time

In [ ]:
# -----------------------------
# NTD-CFE adapted to forecast-band objective
# -----------------------------

class NTDCFEPolicy(nn.Module):
    def __init__(self, input_len: int, n_features: int, actionable_idx: Sequence[int], hidden_dim: int = 128):
        super().__init__()
        self.input_len = input_len
        self.n_features = n_features
        self.actionable_idx = list(actionable_idx)
        self.n_actionable = len(self.actionable_idx)
        flat_dim = input_len * n_features
        self.backbone = nn.Sequential(nn.Linear(flat_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.time_head = nn.Linear(hidden_dim, input_len)
        self.feature_head = nn.Linear(hidden_dim, self.n_actionable)
        self.mu_head = nn.Linear(hidden_dim, self.n_actionable)
        self.log_std = nn.Parameter(torch.zeros(self.n_actionable))

    def forward(self, x_window):
        z = x_window.reshape(x_window.shape[0], -1)
        h = self.backbone(z)
        return self.time_head(h), self.feature_head(h), self.mu_head(h), self.log_std.unsqueeze(0).expand(x_window.shape[0], -1)


def sample_ntd_action(policy: NTDCFEPolicy, current_np: np.ndarray):
    x = torch.tensor(current_np[None, ...], dtype=torch.float32, device=cfg.device)
    time_logits, feat_logits, mu, log_std = policy(x)
    time_dist = torch.distributions.Categorical(logits=time_logits[0])
    feat_dist = torch.distributions.Categorical(logits=feat_logits[0])
    t_idx = time_dist.sample()
    f_local = feat_dist.sample()
    std = torch.exp(log_std[0, f_local]).clamp(1e-3, 2.0)
    val_dist = torch.distributions.Normal(mu[0, f_local], std)
    strength = val_dist.sample()
    log_prob = time_dist.log_prob(t_idx) + feat_dist.log_prob(f_local) + val_dist.log_prob(strength)
    return {"time": int(t_idx.item()), "feature": int(policy.actionable_idx[int(f_local.item())]), "strength": float(strength.item())}, log_prob


def ntd_state_transition(x_scaled: np.ndarray, action: Dict[str, Any], train_scaled: np.ndarray) -> np.ndarray:
    x_new = x_scaled.copy()
    t0, fidx = action["time"], action["feature"]
    delta = cfg.ntd_step_scale * action["strength"]
    x_new[t0:, fidx] = x_new[t0:, fidx] + delta
    # Range constraint as in NTD-CFE optional Crange.
    x_new[:, fidx] = np.clip(x_new[:, fidx], train_scaled[:, fidx].min(), train_scaled[:, fidx].max())
    return x_new.astype(np.float32)


def run_ntdcfe(model: nn.Module, bundle: PreparedDataset, x_query_scaled: np.ndarray,
               upper_scaled: np.ndarray, lower_scaled: np.ndarray,
               upper_orig: np.ndarray, lower_orig: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    start = time.time()
    actionable_idx = bundle.change_feature_idx
    policy = NTDCFEPolicy(cfg.input_len, bundle.n_features, actionable_idx).to(cfg.device)
    opt = torch.optim.Adam(policy.parameters(), lr=cfg.ntd_lr)
    best_cf = x_query_scaled.copy()
    best_pred = predict_model(model, best_cf)
    best_key = (-1.0, float("inf"))

    for _episode in range(cfg.ntd_max_episodes):
        current = x_query_scaled.copy()
        log_probs, rewards = [], []
        for _step in range(cfg.ntd_max_steps_per_episode):
            action, log_prob = sample_ntd_action(policy, current)
            nxt = ntd_state_transition(current, action, bundle.train_scaled)
            pred = predict_model(model, nxt)
            pred_orig = bundle.inverse_target(pred)
            val = validity_ratio_metric(pred_orig, upper_orig, lower_orig)
            prox = proximity_metric(x_query_scaled, nxt)
            band_dist = distance_to_band(pred_orig, upper_orig, lower_orig)
            reward = val - cfg.ntd_lambda_prox * prox - 0.01 * band_dist
            log_probs.append(log_prob)
            rewards.append(float(reward))
            current = nxt
            key = (val, -prox)
            if key > best_key:
                best_key = key
                best_cf = current.copy()
                best_pred = pred.copy()
            if val >= 1.0:
                break
        # REINFORCE update.
        if log_probs:
            returns = []
            G = 0.0
            for r in reversed(rewards):
                G = r + cfg.ntd_gamma * G
                returns.insert(0, G)
            ret = torch.tensor(returns, dtype=torch.float32, device=cfg.device)
            if len(ret) > 1:
                ret = (ret - ret.mean()) / (ret.std() + 1e-8)
            loss = -torch.stack([lp * R for lp, R in zip(log_probs, ret)]).sum()
            opt.zero_grad(); loss.backward(); opt.step()
    return best_cf.astype(np.float32), best_pred.astype(np.float32), time.time() - start

In [ ]:
# -----------------------------
# Genetic Algorithm adapted from Granger + quantile-regressor approach
# -----------------------------

def granger_screen_features(bundle: PreparedDataset) -> pd.DataFrame:
    rows = []
    train = bundle.df.iloc[bundle.train_range[0]:bundle.train_range[1]][bundle.feature_cols].copy()
    for feat in cfg.change_feature_names:
        if feat not in train.columns:
            continue
        try:
            if HAS_STATSMODELS:
                # statsmodels expects [target, candidate] and tests whether candidate Granger-causes target.
                res = grangercausalitytests(train[[cfg.target_col, feat]].dropna(), maxlag=min(4, cfg.ga_lag_order), verbose=False)
                pvals = [res[lag][0]["ssr_ftest"][1] for lag in range(1, min(4, cfg.ga_lag_order) + 1)]
                pval = float(np.min(pvals))
            else:
                pval = 1.0
        except Exception:
            pval = 1.0
        rows.append({"feature": feat, "granger_pvalue": pval})
    out = pd.DataFrame(rows).sort_values("granger_pvalue")
    selected = out[out["granger_pvalue"] <= cfg.ga_granger_alpha]["feature"].tolist()
    if not selected and len(out):
        selected = out.head(min(2, len(out)))["feature"].tolist()
    out["ga_selected"] = out["feature"].isin(selected)
    return out


def make_lagged_supervised(series: np.ndarray, lag: int) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for i in range(lag, len(series)):
        X.append(series[i-lag:i])
        y.append(series[i])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def train_quantile_models(bundle: PreparedDataset, selected_features: Sequence[str]) -> Dict[str, Dict[float, GradientBoostingRegressor]]:
    models: Dict[str, Dict[float, GradientBoostingRegressor]] = {}
    train_orig = bundle.train_original
    for feat in selected_features:
        idx = bundle.feature_cols.index(feat)
        X, y = make_lagged_supervised(train_orig[:, idx], cfg.ga_lag_order)
        models[feat] = {}
        if len(X) < 20:
            continue
        for q in cfg.ga_quantiles:
            m = GradientBoostingRegressor(loss="quantile", alpha=float(q), n_estimators=80, max_depth=3, random_state=cfg.seed)
            m.fit(X, y)
            models[feat][float(q)] = m
    return models


def generate_quantile_suffix(bundle: PreparedDataset, x_scaled: np.ndarray, feat_name: str, q_sequence: Sequence[float], q_models: Dict[str, Dict[float, GradientBoostingRegressor]]) -> np.ndarray:
    idx = bundle.feature_cols.index(feat_name)
    hist_orig = bundle.inverse_window(x_scaled)[:, idx].astype(np.float32).tolist()
    suffix = []
    for q in q_sequence:
        available = q_models.get(feat_name, {})
        q_key = float(q)
        if q_key not in available:
            val = hist_orig[-1]
        else:
            lag_vals = np.asarray(hist_orig[-cfg.ga_lag_order:], dtype=np.float32).reshape(1, -1)
            val = float(available[q_key].predict(lag_vals)[0])
        suffix.append(val)
        hist_orig.append(val)
    return np.asarray(suffix, dtype=np.float32)


def build_ga_cf_window(bundle: PreparedDataset, x_scaled: np.ndarray, individual: np.ndarray, features: Sequence[str], q_models) -> np.ndarray:
    x_orig = bundle.inverse_window(x_scaled)
    x_cf = x_orig.copy()
    L = min(cfg.ga_suffix_len, cfg.input_len)
    editable_time_idx = np.arange(cfg.input_len - L, cfg.input_len)
    cursor = 0
    for feat in features:
        feat_idx = bundle.feature_cols.index(feat)
        q_seq = individual[cursor:cursor+L]
        suffix = generate_quantile_suffix(bundle, x_scaled, feat, q_seq, q_models)
        x_cf[editable_time_idx, feat_idx] = suffix
        cursor += L
    x_cf_scaled = bundle.scale_window(x_cf)
    # Range constraints.
    mins = bundle.train_scaled.min(axis=0); maxs = bundle.train_scaled.max(axis=0)
    for feat in features:
        fi = bundle.feature_cols.index(feat)
        x_cf_scaled[:, fi] = np.clip(x_cf_scaled[:, fi], mins[fi], maxs[fi])
    return x_cf_scaled.astype(np.float32)


def tournament_select(pop: List[np.ndarray], scores: List[float]) -> np.ndarray:
    k = min(cfg.ga_tournament_size, len(pop))
    ids = np.random.choice(len(pop), size=k, replace=False)
    best = min(ids, key=lambda i: scores[i])
    return pop[best].copy()


def crossover(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if len(a) < 3 or np.random.rand() > cfg.ga_crossover_prob:
        return a.copy(), b.copy()
    p1, p2 = sorted(np.random.choice(np.arange(1, len(a)), size=2, replace=False))
    c1, c2 = a.copy(), b.copy()
    c1[p1:p2], c2[p1:p2] = b[p1:p2], a[p1:p2]
    return c1, c2


def mutate(ind: np.ndarray) -> np.ndarray:
    out = ind.copy()
    qs = np.asarray(cfg.ga_quantiles)
    for i in range(len(out)):
        if np.random.rand() < cfg.ga_mutation_prob:
            out[i] = float(np.random.choice(qs))
    return out


def run_ga_cf(model: nn.Module, bundle: PreparedDataset, x_query_scaled: np.ndarray,
              upper_scaled: np.ndarray, lower_scaled: np.ndarray,
              upper_orig: np.ndarray, lower_orig: np.ndarray,
              selected_features: Sequence[str], q_models) -> Tuple[np.ndarray, np.ndarray, float]:
    start = time.time()
    if not selected_features:
        pred = predict_model(model, x_query_scaled)
        return x_query_scaled.copy(), pred, time.time() - start
    L = min(cfg.ga_suffix_len, cfg.input_len)
    chrom_len = len(selected_features) * L
    qs = np.asarray(cfg.ga_quantiles)
    pop = [np.random.choice(qs, size=chrom_len).astype(float) for _ in range(cfg.ga_population)]
    best_x = x_query_scaled.copy()
    best_pred = predict_model(model, best_x)
    best_score = float("inf")

    def fitness(ind):
        nonlocal best_x, best_pred, best_score
        xcf = build_ga_cf_window(bundle, x_query_scaled, ind, selected_features, q_models)
        pred = predict_model(model, xcf)
        pred_orig = bundle.inverse_target(pred)
        val = validity_ratio_metric(pred_orig, upper_orig, lower_orig)
        band = distance_to_band(pred_orig, upper_orig, lower_orig)
        prox = proximity_metric(x_query_scaled, xcf)
        likelihood_pen = float(np.mean(np.abs(ind - 0.5)))
        score = (1.0 - val) + band + cfg.ga_proximity_weight * prox + cfg.ga_likelihood_weight * likelihood_pen
        if score < best_score:
            best_score = score; best_x = xcf.copy(); best_pred = pred.copy()
        return score

    scores = [fitness(ind) for ind in pop]
    for _gen in range(cfg.ga_generations):
        new_pop = []
        elite_idx = int(np.argmin(scores))
        new_pop.append(pop[elite_idx].copy())
        n_imm = int(cfg.ga_immigration_rate * cfg.ga_population)
        while len(new_pop) < cfg.ga_population - n_imm:
            p1 = tournament_select(pop, scores)
            p2 = tournament_select(pop, scores)
            c1, c2 = crossover(p1, p2)
            new_pop.append(mutate(c1))
            if len(new_pop) < cfg.ga_population - n_imm:
                new_pop.append(mutate(c2))
        while len(new_pop) < cfg.ga_population:
            new_pop.append(np.random.choice(qs, size=chrom_len).astype(float))
        pop = new_pop
        scores = [fitness(ind) for ind in pop]
        pred_orig = bundle.inverse_target(best_pred)
        if validity_ratio_metric(pred_orig, upper_orig, lower_orig) >= 1.0:
            break
    return best_x.astype(np.float32), best_pred.astype(np.float32), time.time() - start

## Original seven-method run loop, incremental outputs and all original aggregate plots and tables

In [ ]:
# -----------------------------
# End-to-end benchmark loop, incremental saving, plots, and report
# -----------------------------

METHOD_ORDER = [
    "ForecastCF-target",
    "ForecastCF-exog",
    "COMET-exog",
    "DiffusionCF-target",
    "DiffusionCF-exog",
    "NTD-CFE-exog",
    "GA-exog",
]


def choose_eval_indices(n: int, dataset_name: str) -> np.ndarray:
    all_idx = np.arange(n)
    if cfg.max_test_windows_per_dataset is None or cfg.max_test_windows_per_dataset >= n:
        return all_idx
    rng = np.random.default_rng(int(hashlib.sha256(f"{dataset_name}:{cfg.eval_seed}".encode()).hexdigest()[:8],16))
    return np.sort(rng.choice(all_idx, size=cfg.max_test_windows_per_dataset, replace=False))


def run_one_method(method_name: str, model: nn.Module, bundle: PreparedDataset, x_scaled: np.ndarray,
                   bounds: Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
                   diffusion_cache: Dict[str, Any], ga_cache: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray, float]:
    upper_scaled, lower_scaled, upper_orig, lower_orig = bounds
    if method_name == "ForecastCF-target":
        return make_forecastcf(model, bundle, [bundle.target_idx], method_name).generate(x_scaled, upper_scaled, lower_scaled)
    if method_name == "ForecastCF-exog":
        return make_forecastcf(model, bundle, bundle.change_feature_idx, method_name).generate(x_scaled, upper_scaled, lower_scaled)
    if method_name == "COMET-exog":
        return make_comet(model, bundle, x_scaled).generate(x_scaled, upper_scaled, lower_scaled)
    if method_name == "DiffusionCF-target":
        return diffusion_cache["target"].generate(x_scaled, upper_scaled, lower_scaled)
    if method_name == "DiffusionCF-exog":
        return diffusion_cache["exog"].generate(x_scaled, upper_scaled, lower_scaled)
    if method_name == "NTD-CFE-exog":
        return run_ntdcfe(model, bundle, x_scaled, upper_scaled, lower_scaled, upper_orig, lower_orig)
    if method_name == "GA-exog":
        return run_ga_cf(model, bundle, x_scaled, upper_scaled, lower_scaled, upper_orig, lower_orig,
                         ga_cache["selected_features"], ga_cache["q_models"])
    raise ValueError(method_name)


def save_intermediate(rows: List[Dict[str, Any]], out_csv: Path):
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, index=False)


# -----------------------------
# Optional qualitative counterfactual plots
# -----------------------------

def _sanitize_filename(text: Any) -> str:
    """Make a short filesystem-safe string."""
    keep = []
    for ch in str(text):
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)[:180]


def changed_features_summary(bundle: PreparedDataset, x_scaled: np.ndarray, x_cf_scaled: np.ndarray,
                             tol: float = 1e-6) -> Tuple[List[str], Dict[str, Dict[str, float]]]:
    """Return changed feature names and simple per-feature change statistics in scaled space."""
    diff = np.abs(np.asarray(x_cf_scaled, dtype=np.float32) - np.asarray(x_scaled, dtype=np.float32))
    changed_names: List[str] = []
    stats: Dict[str, Dict[str, float]] = {}
    for j, feat in enumerate(bundle.feature_cols):
        d = diff[:, j]
        changed = d > tol
        if bool(changed.any()):
            changed_names.append(feat)
        stats[feat] = {
            "changed_steps": int(changed.sum()),
            "changed_fraction": float(changed.mean()),
            "mean_abs_change_scaled": float(d.mean()),
            "max_abs_change_scaled": float(d.max()),
        }
    return changed_names, stats


def should_save_cf_example_plot(counter: int, method_name: str, dataset_dir: Path) -> bool:
    """Decide whether to save a qualitative plot for this processed test window and method.

    Defaults are intentionally conservative and controlled with optional cfg attributes:
      cfg.example_plot_every = 10                 # save every 10 processed test windows
      cfg.example_plot_methods = METHOD_ORDER     # methods to plot
      cfg.example_plot_max_per_dataset = None     # optional cap per dataset
      cfg.save_cf_example_plots = True            # switch off plotting entirely
    """
    if not bool(getattr(cfg, "save_cf_example_plots", True)):
        return False
    every = int(getattr(cfg, "example_plot_every", 20))
    if every <= 0 or counter % every != 0:
        return False
    methods = list(getattr(cfg, "example_plot_methods", METHOD_ORDER))
    if method_name not in methods:
        return False
    max_per_dataset = getattr(cfg, "example_plot_max_per_dataset", None)
    if max_per_dataset is not None:
        plot_dir = dataset_dir / "cf_example_plots"
        existing = list(plot_dir.glob("*.png")) if plot_dir.exists() else []
        if len(existing) >= int(max_per_dataset):
            return False
    return True


def plot_counterfactual_example(bundle: PreparedDataset,
                                method_name: str,
                                x_scaled: np.ndarray,
                                x_cf_scaled: np.ndarray,
                                y_true_orig: np.ndarray,
                                pred0_orig: np.ndarray,
                                pred_cf_scaled: np.ndarray,
                                upper_orig: np.ndarray,
                                lower_orig: np.ndarray,
                                test_idx: int,
                                start_idx: int,
                                dataset_dir: Path,
                                counter: Optional[int] = None,
                                tol: float = 1e-6) -> Optional[Path]:
    """Plot one qualitative counterfactual example and save it as PNG.

    The plot contains:
      1) historical original vs counterfactual values for every input feature;
      2) forecast horizon with true future, original forecast, counterfactual forecast, and desired bounds;
      3) a title listing which features actually changed.

    It is deliberately wrapped by callers in try/except so plotting can never stop the benchmark.
    """
    plot_dir = safe_mkdir(dataset_dir / "cf_example_plots")

    x_orig = bundle.inverse_window(np.asarray(x_scaled, dtype=np.float32))
    x_cf_orig = bundle.inverse_window(np.asarray(x_cf_scaled, dtype=np.float32))
    pred_cf_orig = bundle.inverse_target(np.asarray(pred_cf_scaled, dtype=np.float32))

    changed_names, stats = changed_features_summary(bundle, x_scaled, x_cf_scaled, tol=tol)
    changed_text = ", ".join(changed_names) if changed_names else "none"

    n_features = len(bundle.feature_cols)
    n_rows = n_features + 1
    fig_height = max(8.0, 2.4 * n_rows)
    fig, axes = plt.subplots(n_rows, 1, figsize=(13, fig_height), sharex=False)
    axes = np.atleast_1d(axes)

    t_hist = np.arange(x_orig.shape[0])
    for j, feat in enumerate(bundle.feature_cols):
        ax = axes[j]
        ax.plot(t_hist, x_orig[:, j], label="Original history", linewidth=1.6)
        ax.plot(t_hist, x_cf_orig[:, j], label="Counterfactual history", linestyle="--", linewidth=1.6)
        ax.set_ylabel(feat)
        ax.set_title(f"Historical input: {feat}")
        ax.grid(True, alpha=0.25)
        if j == 0:
            ax.legend(loc="best", fontsize=9)

    t_fore = np.arange(1, len(pred0_orig) + 1)
    ax = axes[-1]
    ax.plot(t_fore, y_true_orig, label="True future", linewidth=1.6, alpha=0.85)
    ax.plot(t_fore, pred0_orig, label="Original forecast", linewidth=1.8)
    ax.plot(t_fore, pred_cf_orig, label="Counterfactual forecast", linestyle="--", linewidth=1.8)
    ax.plot(t_fore, upper_orig, label="Upper bound", linestyle=":", linewidth=2.0)
    ax.plot(t_fore, lower_orig, label="Lower bound", linestyle=":", linewidth=2.0)
    ax.fill_between(t_fore, lower_orig, upper_orig, alpha=0.12)
    ax.set_xlabel("Forecast horizon step")
    ax.set_ylabel(bundle.feature_cols[bundle.target_idx])
    ax.set_title("Forecast horizon and desired bounds")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=9)

    counter_txt = f" | processed window #{counter}" if counter is not None else ""
    fig.suptitle(
        f"{bundle.name} | {method_name} | test_idx={test_idx} | start_idx={start_idx}{counter_txt}\n"
        f"Changed features: {changed_text}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.955))

    filename = _sanitize_filename(f"{bundle.name}__{method_name}__counter{counter}__test{test_idx}__start{start_idx}.png")
    out_path = plot_dir / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    # Also append a tiny audit CSV next to the plots. This makes it easy to check changed variables later.
    audit_path = plot_dir / "cf_example_plot_audit.csv"
    audit_row = {
        "dataset": bundle.name,
        "method": method_name,
        "counter": counter,
        "test_idx": int(test_idx),
        "start_idx": int(start_idx),
        "plot_path": str(out_path),
        "changed_features": changed_text,
    }
    for feat, feat_stats in stats.items():
        safe_feat = _sanitize_filename(feat)
        audit_row[f"{safe_feat}_changed_steps"] = feat_stats["changed_steps"]
        audit_row[f"{safe_feat}_mean_abs_change_scaled"] = feat_stats["mean_abs_change_scaled"]
        audit_row[f"{safe_feat}_max_abs_change_scaled"] = feat_stats["max_abs_change_scaled"]
    if audit_path.exists():
        old = pd.read_csv(audit_path)
        pd.concat([old, pd.DataFrame([audit_row])], ignore_index=True).to_csv(audit_path, index=False)
    else:
        pd.DataFrame([audit_row]).to_csv(audit_path, index=False)

    return out_path



# Public high-dimensional plotting adaptation ONLY; all original methods and metrics remain unchanged.
# The source plots n_features+1 axes in one PNG; 321 Electricity channels exceed image backend
# height limits. Render EVERY channel in numbered pages, each with the SAME original-vs-CF
# history and forecast+desired-bounds panel. For <=32 features use original function verbatim.
_orig_plot_counterfactual_example = plot_counterfactual_example

def plot_counterfactual_example(bundle: PreparedDataset,
                                method_name: str,
                                x_scaled: np.ndarray,
                                x_cf_scaled: np.ndarray,
                                y_true_orig: np.ndarray,
                                pred0_orig: np.ndarray,
                                pred_cf_scaled: np.ndarray,
                                upper_orig: np.ndarray,
                                lower_orig: np.ndarray,
                                test_idx: int,
                                start_idx: int,
                                dataset_dir: Path,
                                counter: Optional[int] = None,
                                tol: float = 1e-6) -> Optional[Path]:
    if bundle.n_features <= 32:
        return _orig_plot_counterfactual_example(bundle, method_name, x_scaled, x_cf_scaled,
            y_true_orig, pred0_orig, pred_cf_scaled, upper_orig, lower_orig,
            test_idx, start_idx, dataset_dir, counter, tol)
    page_size = 24
    plot_dir = safe_mkdir(dataset_dir / 'cf_example_plots')
    x_orig = bundle.inverse_window(np.asarray(x_scaled, dtype=np.float32))
    x_cf_orig = bundle.inverse_window(np.asarray(x_cf_scaled, dtype=np.float32))
    pred_cf_orig = bundle.inverse_target(np.asarray(pred_cf_scaled, dtype=np.float32))
    changed_names, stats = changed_features_summary(bundle, x_scaled, x_cf_scaled, tol=tol)
    changed_text = ', '.join(changed_names) if changed_names else 'none'
    saved=[]
    for page_num, start in enumerate(range(0,bundle.n_features,page_size),start=1):
        inds=list(range(start,min(start+page_size,bundle.n_features)))
        fig,axes=plt.subplots(len(inds)+1,1,figsize=(13,max(8,2.4*(len(inds)+1))),sharex=False)
        axes=np.atleast_1d(axes)
        t_hist=np.arange(x_orig.shape[0])
        for k,j in enumerate(inds):
            feat=bundle.feature_cols[j];ax=axes[k]
            ax.plot(t_hist,x_orig[:,j],label='Original history',linewidth=1.6)
            ax.plot(t_hist,x_cf_orig[:,j],label='Counterfactual history',linestyle='--',linewidth=1.6)
            ax.set_ylabel(feat);ax.set_title(f'Historical input: {feat}');ax.grid(True,alpha=.25)
            if k==0:ax.legend(loc='best',fontsize=9)
        ax=axes[-1];t_fore=np.arange(1,len(pred0_orig)+1)
        ax.plot(t_fore,y_true_orig,label='True future',linewidth=1.6,alpha=.85)
        ax.plot(t_fore,pred0_orig,label='Original forecast',linewidth=1.8)
        ax.plot(t_fore,pred_cf_orig,label='Counterfactual forecast',linestyle='--',linewidth=1.8)
        ax.plot(t_fore,upper_orig,label='Upper bound',linestyle=':',linewidth=2)
        ax.plot(t_fore,lower_orig,label='Lower bound',linestyle=':',linewidth=2)
        ax.fill_between(t_fore,lower_orig,upper_orig,alpha=.12)
        ax.set_xlabel('Forecast horizon step');ax.set_ylabel(bundle.feature_cols[bundle.target_idx])
        ax.set_title('Forecast horizon and desired bounds');ax.grid(True,alpha=.25)
        ax.legend(loc='best',fontsize=9)
        fig.suptitle(f'{bundle.name} | {method_name} | test_idx={test_idx} | start_idx={start_idx} | '
                     f'page={page_num} | changed features: {changed_text[:200]}',fontsize=12)
        fig.tight_layout(rect=(0,0,1,.965))
        stem=_sanitize_filename(f'{bundle.name}__{method_name}__counter{counter}__test{test_idx}__start{start_idx}')
        out_path=plot_dir/f'{stem}__page{page_num:03d}.png'
        fig.savefig(out_path,dpi=160,bbox_inches='tight');plt.close(fig)
        saved.append(out_path)
    audit_path=plot_dir/'cf_example_plot_audit.csv'
    audit_row={'dataset':bundle.name,'method':method_name,'counter':counter,
               'test_idx':int(test_idx),'start_idx':int(start_idx),
               'plot_path':str(saved[0]),'all_plot_pages':' | '.join(map(str,saved)),
               'changed_features':changed_text}
    for feat,feat_stats in stats.items():
        key=_sanitize_filename(feat)
        audit_row[f'{key}_changed_steps']=feat_stats['changed_steps']
        audit_row[f'{key}_mean_abs_change_scaled']=feat_stats['mean_abs_change_scaled']
        audit_row[f'{key}_max_abs_change_scaled']=feat_stats['max_abs_change_scaled']
    old=pd.read_csv(audit_path) if audit_path.exists() else pd.DataFrame()
    pd.concat([old,pd.DataFrame([audit_row])],ignore_index=True).to_csv(audit_path,index=False)
    return saved[0]

def run_dataset_benchmark(csv_path: Path, results_dir: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    bundle = prepare_dataset(csv_path)
    dataset_dir = safe_mkdir(results_dir / "datasets" / bundle.name)
    cf_csv = dataset_dir / "counterfactual_results.csv"
    forecast_json = dataset_dir / "forecast_metrics.json"

    model = load_best_gru(bundle)
    forecast_metrics = evaluate_forecaster(model, bundle)
    forecast_metrics.update({"dataset": bundle.name, "csv_path": str(csv_path), "model_path": str(find_saved_model_path(bundle.name))})
    forecast_json.write_text(json.dumps(forecast_metrics, indent=2), encoding="utf-8")

    # Pre-train method-specific reusable components once per dataset.
    print(f"  Training DiffusionCF-target components for {bundle.name}...")
    vae_target = train_vae_for(bundle, [bundle.target_idx])
    ddpm_target = train_ddpm_for(bundle, [bundle.target_idx])
    print(f"  Training DiffusionCF-exog components for {bundle.name}...")
    vae_exog = train_vae_for(bundle, bundle.change_feature_idx)
    ddpm_exog = train_ddpm_for(bundle, bundle.change_feature_idx)
    diffusion_cache = {
        "target": DiffusionCFAdapter(model, [bundle.target_idx], vae_target, ddpm_target, bundle.train_scaled, "DiffusionCF-target"),
        "exog": DiffusionCFAdapter(model, bundle.change_feature_idx, vae_exog, ddpm_exog, bundle.train_scaled, "DiffusionCF-exog"),
    }

    granger_df = granger_screen_features(bundle)
    granger_df.to_csv(dataset_dir / "ga_granger_screen.csv", index=False)
    selected_features = granger_df.loc[granger_df["ga_selected"], "feature"].tolist()
    print(f"  GA selected features for {bundle.name}: {selected_features}")
    q_models = train_quantile_models(bundle, selected_features)
    ga_cache = {"selected_features": selected_features, "q_models": q_models}

    # Resume support.
    done_keys = set()
    rows: List[Dict[str, Any]] = []
    if cfg.resume_existing_results and cf_csv.exists():
        old = pd.read_csv(cf_csv)
        rows = old.to_dict("records")
        done_keys = set(zip(old["test_idx"].astype(int), old["method"].astype(str)))
        print(f"  Resuming {bundle.name}: {len(done_keys)} existing method-window results.")

    eval_indices = choose_eval_indices(len(bundle.test_ds), bundle.name)
    print(f"  Evaluating {len(eval_indices)} test windows for {bundle.name}.")

    for counter, idx in enumerate(eval_indices, start=1):
        x_tensor, y_tensor, start_idx = bundle.test_ds[int(idx)]
        x_scaled = x_tensor.numpy().astype(np.float32)
        y_true_scaled = y_tensor.numpy().astype(np.float32)
        pred0_scaled = predict_model(model, x_scaled)
        upper_scaled, lower_scaled, upper_orig, lower_orig = generate_comet_style_bounds(bundle, x_scaled)
        pred0_orig = bundle.inverse_target(pred0_scaled)
        y_true_orig = bundle.inverse_target(y_true_scaled)
        baseline = {
            "baseline_validity_ratio": validity_ratio_metric(pred0_orig, upper_orig, lower_orig),
            "baseline_stepwise_validity_auc": stepwise_validity_auc_metric(pred0_orig, upper_orig, lower_orig),
            "baseline_consecutive_valid_steps": prefix_consecutive_valid_steps(pred0_orig, upper_orig, lower_orig),
            "baseline_distance_to_band": distance_to_band(pred0_orig, upper_orig, lower_orig),
            "true_mae_vs_forecast": float(np.mean(np.abs(y_true_orig - pred0_orig))),
        }
        for method in METHOD_ORDER:
            key = (int(idx), method)
            if key in done_keys:
                continue
            try:
                x_cf, pred_cf_scaled, runtime = run_one_method(method, model, bundle, x_scaled,
                                                               (upper_scaled, lower_scaled, upper_orig, lower_orig),
                                                               diffusion_cache, ga_cache)
                metrics = evaluate_cf(bundle, x_scaled, x_cf, pred_cf_scaled, upper_orig, lower_orig, runtime)
                status = "ok"
                error_msg = ""
                if should_save_cf_example_plot(counter, method, dataset_dir):
                    try:
                        plot_counterfactual_example(
                            bundle=bundle,
                            method_name=method,
                            x_scaled=x_scaled,
                            x_cf_scaled=x_cf,
                            y_true_orig=y_true_orig,
                            pred0_orig=pred0_orig,
                            pred_cf_scaled=pred_cf_scaled,
                            upper_orig=upper_orig,
                            lower_orig=lower_orig,
                            test_idx=int(idx),
                            start_idx=int(start_idx),
                            dataset_dir=dataset_dir,
                            counter=counter,
                        )
                    except Exception as plot_exc:
                        print(f"      Plot warning for {bundle.name} {method} idx={idx}: {plot_exc}")
            except Exception as exc:
                x_cf = x_scaled.copy()
                pred_cf_scaled = pred0_scaled.copy()
                metrics = evaluate_cf(bundle, x_scaled, x_cf, pred_cf_scaled, upper_orig, lower_orig, 0.0)
                status = "error"
                error_msg = repr(exc)
            pred_cf_orig_for_horizon = bundle.inverse_target(pred_cf_scaled)
            horizon_validity = per_step_validity_dict(
                pred_cf_orig_for_horizon,
                upper_orig,
                lower_orig,
                prefix="valid_h",
            )

            rows.append({
                "dataset": bundle.name,
                "test_idx": int(idx),
                "start_idx": int(start_idx),
                "method": method,
                "status": status,
                "error": error_msg,
                **baseline,
                **metrics,
                **horizon_validity,
            })
            save_intermediate(rows, cf_csv)
        if counter % 10 == 0:
            print(f"    {bundle.name}: {counter}/{len(eval_indices)} windows processed")
    df = pd.DataFrame(rows)
    df.to_csv(cf_csv, index=False)
    return df, forecast_metrics


def aggregate_and_plot(results_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_files = list((results_dir / "datasets").glob("*/counterfactual_results.csv"))
    if not all_files:
        raise FileNotFoundError("No counterfactual_results.csv files found.")

    raw = pd.concat([pd.read_csv(p) for p in all_files], ignore_index=True)
    raw.to_csv(results_dir / "all_counterfactual_results_raw.csv", index=False)

    metric_cols = [
        "validity_ratio",
        "stepwise_validity_auc",
        "consecutive_valid_steps",
        "max_consecutive_valid_steps",
        "distance_to_band",
        "proximity_l2_scaled",
        "proximity_l1_scaled",
        "sparsity_count",
        "sparsity_ratio",
        "runtime_sec",
    ]

    # Keep the original mean-only files for backwards compatibility.
    summary = raw.groupby(["dataset", "method"], as_index=False)[metric_cols].mean()
    overall = raw.groupby(["method"], as_index=False)[metric_cols].mean()
    summary.to_csv(results_dir / "counterfactual_summary_by_dataset_method.csv", index=False)
    overall.to_csv(results_dir / "counterfactual_summary_overall.csv", index=False)

    # New richer statistics: mean/std/min/max/count.
    def _stats_frame(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
        stats = df.groupby(group_cols)[metric_cols].agg(["mean", "std", "min", "max", "count"]).reset_index()
        stats.columns = [
            "_".join([str(x) for x in col if str(x) != ""]).rstrip("_")
            if isinstance(col, tuple) else str(col)
            for col in stats.columns
        ]
        # std is NaN when a group has one sample; use 0 so plotting does not fail.
        for c in stats.columns:
            if c.endswith("_std"):
                stats[c] = stats[c].fillna(0.0)
        return stats

    summary_stats = _stats_frame(raw, ["dataset", "method"])
    overall_stats = _stats_frame(raw, ["method"])
    summary_stats.to_csv(results_dir / "counterfactual_summary_by_dataset_method_stats.csv", index=False)
    overall_stats.to_csv(results_dir / "counterfactual_summary_overall_stats.csv", index=False)

    plots_dir = safe_mkdir(results_dir / "plots")

    def _metric_stats_for_plot(metric: str) -> pd.DataFrame:
        cols = ["method", f"{metric}_mean", f"{metric}_std", f"{metric}_min", f"{metric}_max", f"{metric}_count"]
        tmp = overall_stats[cols].copy()
        tmp = tmp.set_index("method").reindex(METHOD_ORDER).reset_index()
        for c in cols[1:]:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        return tmp

    def _plot_overall_bar(metric: str, log_y: bool = False, error_style: str = "std") -> None:
        """Plot mean metric per method with variability.

        error_style="std" shows mean ± one standard deviation, which is usually easier to read
        than min/max because min/max can be dominated by a single outlier. The corresponding
        min/max values are still saved in the *_stats.csv files.
        """
        tmp = _metric_stats_for_plot(metric)
        x = np.arange(len(tmp))
        mean = tmp[f"{metric}_mean"].to_numpy(dtype=float)
        std = tmp[f"{metric}_std"].fillna(0).to_numpy(dtype=float)
        vmin = tmp[f"{metric}_min"].to_numpy(dtype=float)
        vmax = tmp[f"{metric}_max"].to_numpy(dtype=float)

        plt.figure(figsize=(11, 5.5))

        if log_y:
            # Matplotlib log axes cannot display zero. Runtime should be positive for successful
            # counterfactuals, but error fallbacks may have 0. We plot those at a tiny floor and
            # keep the exact values in the saved CSV.
            positive_values = mean[np.isfinite(mean) & (mean > 0)]
            floor = max(float(np.nanmin(positive_values)) / 20.0, 1e-6) if len(positive_values) else 1e-6
            plot_mean = np.where(np.isfinite(mean) & (mean > 0), mean, floor)

            if error_style == "minmax":
                low_target = np.where(np.isfinite(vmin) & (vmin > 0), vmin, floor)
                high_target = np.where(np.isfinite(vmax) & (vmax > 0), vmax, plot_mean)
                yerr_low = np.maximum(plot_mean - low_target, 0.0)
                yerr_high = np.maximum(high_target - plot_mean, 0.0)
                yerr = np.vstack([yerr_low, yerr_high])
                error_label = "min/max"
            else:
                yerr_low = np.minimum(std, np.maximum(plot_mean - floor, 0.0))
                yerr_high = std
                yerr = np.vstack([yerr_low, yerr_high])
                error_label = "±1 std"

            plt.bar(x, plot_mean, yerr=yerr, capsize=4)
            plt.yscale("log")
            plt.ylabel(f"{metric} (log scale)")
            subtitle = f"Overall mean {metric} with {error_label}; y-axis is log-scaled"
        else:
            if error_style == "minmax":
                yerr = np.vstack([
                    np.maximum(mean - vmin, 0.0),
                    np.maximum(vmax - mean, 0.0),
                ])
                error_label = "min/max"
            else:
                yerr = std
                error_label = "±1 std"

            plt.bar(x, mean, yerr=yerr, capsize=4)
            plt.ylabel(metric)
            subtitle = f"Overall mean {metric} with {error_label}"

        plt.xticks(x, tmp["method"], rotation=35, ha="right")
        plt.title(subtitle)
        plt.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        suffix = "_logy" if log_y else ""
        plt.savefig(plots_dir / f"overall_{metric}{suffix}.png", dpi=220)
        plt.close()

        # For metrics where min/max is useful, also save a min/max version as a separate file.
        if error_style != "minmax":
            plt.figure(figsize=(11, 5.5))
            if log_y:
                positive_values = mean[np.isfinite(mean) & (mean > 0)]
                floor = max(float(np.nanmin(positive_values)) / 20.0, 1e-6) if len(positive_values) else 1e-6
                plot_mean = np.where(np.isfinite(mean) & (mean > 0), mean, floor)
                low_target = np.where(np.isfinite(vmin) & (vmin > 0), vmin, floor)
                high_target = np.where(np.isfinite(vmax) & (vmax > 0), vmax, plot_mean)
                yerr = np.vstack([
                    np.maximum(plot_mean - low_target, 0.0),
                    np.maximum(high_target - plot_mean, 0.0),
                ])
                plt.bar(x, plot_mean, yerr=yerr, capsize=4)
                plt.yscale("log")
                plt.ylabel(f"{metric} (log scale)")
                plt.title(f"Overall mean {metric} with min/max bounds; y-axis is log-scaled")
                minmax_suffix = "_minmax_logy"
            else:
                yerr = np.vstack([
                    np.maximum(mean - vmin, 0.0),
                    np.maximum(vmax - mean, 0.0),
                ])
                plt.bar(x, mean, yerr=yerr, capsize=4)
                plt.ylabel(metric)
                plt.title(f"Overall mean {metric} with min/max bounds")
                minmax_suffix = "_minmax"

            plt.xticks(x, tmp["method"], rotation=35, ha="right")
            plt.grid(axis="y", alpha=0.25)
            plt.tight_layout()
            plt.savefig(plots_dir / f"overall_{metric}{minmax_suffix}.png", dpi=220)
            plt.close()

    # Overall bar plots with variability.
    for metric in [
        "validity_ratio",
        "stepwise_validity_auc",
        "consecutive_valid_steps",
        "proximity_l2_scaled",
        "sparsity_ratio",
        "runtime_sec",
    ]:
        _plot_overall_bar(metric, log_y=(metric == "runtime_sec"), error_style="std")

    # Dataset-method heatmaps remain mean heatmaps. Runtime heatmap uses log1p values so slow
    # methods do not hide differences among fast methods.
    for metric in ["validity_ratio", "proximity_l2_scaled", "runtime_sec", "sparsity_ratio"]:
        pivot = summary.pivot(index="dataset", columns="method", values=metric).reindex(columns=METHOD_ORDER)
        values = pivot.values.astype(float)
        label = metric
        if metric == "runtime_sec":
            values = np.log10(np.maximum(values, 1e-6))
            label = "log10(runtime_sec)"
        plt.figure(figsize=(12, max(5, 0.35 * len(pivot))))
        plt.imshow(values, aspect="auto")
        plt.colorbar(label=label)
        plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha="right")
        plt.yticks(range(len(pivot.index)), pivot.index)
        plt.title(f"Dataset-method heatmap: {label}")
        plt.tight_layout()
        heatmap_name = f"heatmap_{metric}_log10.png" if metric == "runtime_sec" else f"heatmap_{metric}.png"
        plt.savefig(plots_dir / heatmap_name, dpi=220)
        plt.close()

    # Trade-off plot with method-level standard deviation on both axes.
    trade = overall_stats.set_index("method").reindex(METHOD_ORDER).reset_index()
    plt.figure(figsize=(8.5, 6.5))
    x = trade["proximity_l2_scaled_mean"].to_numpy(dtype=float)
    y = trade["validity_ratio_mean"].to_numpy(dtype=float)
    xerr = trade["proximity_l2_scaled_std"].fillna(0).to_numpy(dtype=float)
    yerr = trade["validity_ratio_std"].fillna(0).to_numpy(dtype=float)
    plt.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="o", capsize=3)
    for _, r in trade.iterrows():
        plt.text(r["proximity_l2_scaled_mean"], r["validity_ratio_mean"], r["method"], fontsize=8)
    plt.xlabel("Mean proximity L2 scaled, lower is better")
    plt.ylabel("Mean validity ratio, higher is better")
    plt.title("Validity-proximity trade-off with ±1 std")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / "tradeoff_validity_proximity_with_std.png", dpi=220)
    plt.close()

    # Average ranks: higher validity/stepwise/consecutive better; lower proximity/sparsity/runtime better.
    rank_rows = []
    for dataset, grp in summary.groupby("dataset"):
        for metric, ascending in [
            ("validity_ratio", False),
            ("stepwise_validity_auc", False),
            ("consecutive_valid_steps", False),
            ("proximity_l2_scaled", True),
            ("sparsity_ratio", True),
            ("runtime_sec", True),
        ]:
            ranks = grp.set_index("method")[metric].rank(ascending=ascending, method="average")
            for method, rank in ranks.items():
                rank_rows.append({"dataset": dataset, "metric": metric, "method": method, "rank": rank})
    ranks_df = pd.DataFrame(rank_rows)
    avg_ranks = ranks_df.groupby(["method", "metric"], as_index=False)["rank"].mean()
    avg_ranks.to_csv(results_dir / "average_ranks_by_metric.csv", index=False)

    plt.figure(figsize=(10, 5))
    tmp = avg_ranks.groupby("method")["rank"].mean().reindex(METHOD_ORDER)
    plt.bar(tmp.index, tmp.values)
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Average rank across metrics and datasets, lower is better")
    plt.title("Average-rank summary")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / "average_rank_summary.png", dpi=220)
    # Keep the old filename too so any existing report references do not break.
    plt.savefig(plots_dir / "critical_rank_summary.png", dpi=220)
    plt.close()

    return raw, overall
def write_method_report(results_dir: Path) -> Path:
    text = f"""# EDBT counterfactual benchmark report

Generated: {now_stamp()}

## Goal

The goal of this notebook is to compare ForecastCF, COMET, DiffusionCF, NTD-CFE, and Genetic-Algorithm counterfactual search under a unified multivariate time-series forecasting setup. All methods explain the same saved GRU forecaster for each public dataset. The desired outcome is expressed using COMET-style lower and upper forecast bounds generated separately for each test window.

## Data and forecasting model

The notebook uses the same chronological split as the all-piezometers notebook: train ratio `{cfg.train_ratio}`, validation ratio `{cfg.val_ratio}`, and the remaining suffix as test, with overlap equal to `input_len + horizon` to form valid windows at split boundaries. The input window length is `{cfg.input_len}` and the forecast horizon is `{cfg.horizon}`. Each GRU receives the full multivariate history and predicts the future `{cfg.target_col}` trajectory.

The notebook does not retrain forecasters. It reloads `best_gru_forecaster.pth` for each public dataset from `{cfg.previous_output_root / 'datasets'}`.

## Desired bounds

For every test window, bounds are generated by a COMET-style trend: start from the last historical target value, move toward a configurable desired center over `{cfg.bound_reach_steps}` steps, trim to the `{cfg.horizon}`-step horizon, and add a band width based on the historical target standard deviation. Since these public datasets do not use glucose clinical thresholds, the default desired center is defined as a relative change of `{cfg.desired_change_percent}` from the last historical target value.

## Implemented methods

### ForecastCF-target

This is the faithful ForecastCF adaptation. ForecastCF was originally designed for univariate forecasting and perturbs the historical target series. Here, the GRU still receives all variables, but the optimization mask only allows changes to the historical `{cfg.target_col}` channel. Exogenous channels remain fixed.

### ForecastCF-exog

This is the fairness adaptation. It keeps the same ForecastCF gradient objective but allows changes to the exogenous variables `{cfg.change_feature_names}`. It has no COMET clipping, activity, or historical-value restrictions, so it can be compared against COMET under the same editable variable set.

### COMET-exog

This follows the COMET multivariate-to-univariate intervention idea: keep the target history fixed and change exogenous/actionable variables. The implementation includes clipping to training min/max ranges, an activity mask, and a historical-value penalty. The original activity implementation is retained: names containing rain/precip are zero-inactive; the three specified Weather edit channels are treated as continuous and all other non-precipitation names carry zero activity weights. These are not validated physical actionability constraints.

### DiffusionCF-target

This is the faithful semantic adaptation of DiffusionCF's univariate segment-replacement idea. It trains a VAE detector and DDPM sampler on target-history windows, ranks anomalous target windows, replaces selected windows with generated candidates, and keeps the candidate that best satisfies the forecast bounds.

### DiffusionCF-exog

This is the fairness adaptation of DiffusionCF. The VAE and DDPM are trained on exogenous windows and replace local exogenous segments. This makes the editable variables comparable to COMET, NTD-CFE, and GA. It remains an adaptation because the original DiffusionCF uses block-maxima forecasting, a GEV-based target, and CSDI conditional imputation.

### NTD-CFE-exog

This adapts the RL method to forecasting. The action remains faithful to NTD-CFE: choose an intervention time, choose an actionable feature, and choose a continuous strength. The transition changes the chosen feature from the selected time to the end of the input window, applies range constraints, and evaluates the modified sample through the GRU. Instead of binary class success, the reward uses forecast validity ratio under the shared bounds, plus proximity penalty.

### GA-exog

This adapts the Granger + quantile + genetic algorithm method. Granger screening selects exogenous variables with statistical evidence for predicting the target. Quantile regressors learn plausible values for selected features. A chromosome encodes quantile choices for each selected variable and suffix timestep. Crossover exchanges random chromosome segments, mutation replaces genes only with allowed quantile values, and fitness combines forecast-band validity, distance to the band, proximity, and a small likelihood penalty favoring central quantiles.

## Evaluation metrics

The notebook reports mean results over the selected test windows for each public dataset and method:

- validity ratio: fraction of forecast horizon points inside the desired bounds.
- stepwise validity AUC: prefix consecutive validity normalized by horizon.
- consecutive valid steps: number of valid steps from the start of the horizon.
- max consecutive valid steps: longest valid run anywhere in the horizon.
- proximity: L2 and L1 distances in scaled input space.
- sparsity: number and ratio of changed input entries.
- runtime: seconds per generated counterfactual.

## Important limitations

1. The notebook is a controlled comparative adaptation, not a byte-for-byte reproduction of the original repositories.
2. DiffusionCF is the most heavily adapted because the original task is block-maxima forecasting with a GEV target and CSDI imputation.
3. The COMET activity constraint is carried over from the piezometer code, and public-domain features do not have the same clinical semantics as original COMET treatment variables.
4. Running on the full test set can be computationally expensive, especially for NTD-CFE, GA, and DiffusionCF.
5. The GA implementation uses classical Granger screening for local runtime and interpretability. If exact repository-level faithfulness is required, replace `granger_screen_features` with the repository's LSTM-based Granger test.
"""
    path = results_dir / "method_and_experiment_report.md"
    path.write_text(text, encoding="utf-8")
    return path


def run_all_experiments() -> Path:
    results_dir = safe_mkdir(cfg.results_root)
    safe_mkdir(results_dir / "datasets")
    safe_mkdir(results_dir / "plots")
    (results_dir / "config.json").write_text(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, indent=2), encoding="utf-8")

    paths, missing = discover_dataset_files(cfg.data_root, cfg.datasets)
    print("Found datasets:", [p.stem for p in paths])
    if missing:
        print("Missing datasets:", missing)
    if not paths:
        raise FileNotFoundError(f"No requested piezometer csv files under {cfg.data_root}")

    forecast_rows = []
    for p in paths:
        print(f"\n=== {p.stem} ===")
        try:
            cf_df, forecast_metrics = run_dataset_benchmark(p, results_dir)
            forecast_rows.append(forecast_metrics)
        except Exception as exc:
            print(f"ERROR for {p.stem}: {exc}")
            forecast_rows.append({"dataset": p.stem, "error": repr(exc)})
        pd.DataFrame(forecast_rows).to_csv(results_dir / "forecast_recheck_metrics.csv", index=False)

    raw, overall = aggregate_and_plot(results_dir)
    report_path = write_method_report(results_dir)
    print("\nDone.")
    print("Raw results:", results_dir / "all_counterfactual_results_raw.csv")
    print("Overall summary:", results_dir / "counterfactual_summary_overall.csv")
    print("Report:", report_path)
    return results_dir

## Original boxplots, per-horizon validity, and critical-difference diagrams (functions only; invoked automatically after each dataset run)

In [ ]:
# -----------------------------
# Box-and-whisker plots for all evaluation metrics
# Mean = green triangle
# Median = orange line
# -----------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path


def load_raw_counterfactual_results(results_dir: Path) -> pd.DataFrame:
    """
    Load raw per-sample counterfactual results.

    Uses all_counterfactual_results_raw.csv if it exists.
    Otherwise rebuilds it from datasets/*/counterfactual_results.csv.
    """
    raw_path = results_dir / "all_counterfactual_results_raw.csv"

    if raw_path.exists():
        raw = pd.read_csv(raw_path)
    else:
        all_files = list((results_dir / "datasets").glob("*/counterfactual_results.csv"))

        if not all_files:
            raise FileNotFoundError(
                f"No counterfactual_results.csv files found under {results_dir / 'datasets'}"
            )

        raw = pd.concat([pd.read_csv(p) for p in all_files], ignore_index=True)
        raw.to_csv(raw_path, index=False)

    return raw


def make_metric_boxplots(results_dir: Path) -> None:
    raw = load_raw_counterfactual_results(results_dir)

    # Keep only successful counterfactual rows if the status column exists.
    if "status" in raw.columns:
        print("Status counts:")
        print(raw["status"].value_counts(dropna=False))
        raw_plot = raw[raw["status"].astype(str) == "ok"].copy()
    else:
        raw_plot = raw.copy()

    plots_dir = results_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    metric_cols = [
        "validity_ratio",
        "stepwise_validity_auc",
        "consecutive_valid_steps",
        "max_consecutive_valid_steps",
        "distance_to_band",
        "proximity_l2_scaled",
        "proximity_l1_scaled",
        "sparsity_count",
        "sparsity_ratio",
        "runtime_sec",
    ]

    available_metrics = [m for m in metric_cols if m in raw_plot.columns]

    methods = [m for m in METHOD_ORDER if m in raw_plot["method"].unique()]

    mean_color = "#2ca02c"     # green
    median_color = "#ff7f0e"   # orange

    for metric in available_metrics:
        data = [
            pd.to_numeric(
                raw_plot.loc[raw_plot["method"] == method, metric],
                errors="coerce"
            ).dropna().values
            for method in methods
        ]

        if all(len(values) == 0 for values in data):
            print(f"Skipping {metric}: no valid data.")
            continue

        fig, ax = plt.subplots(figsize=(12, 5.8))

        ax.boxplot(
            data,
            labels=methods,
            showmeans=True,
            showfliers=True,

            # This makes the mean a real green triangle.
            meanprops=dict(
                marker="^",
                markerfacecolor=mean_color,
                markeredgecolor=mean_color,
                markersize=7,
            ),

            # This makes the median a real orange line.
            medianprops=dict(
                color=median_color,
                linewidth=1.8,
            ),
        )

        ax.set_xticklabels(methods, rotation=35, ha="right")
        ax.set_ylabel(metric)
        ax.set_title(f"Distribution of {metric} by method")
        ax.grid(axis="y", alpha=0.25)

        # Axis limits for bounded metrics.
        if metric in ["validity_ratio", "stepwise_validity_auc", "sparsity_ratio"]:
            ax.set_ylim(-0.05, 1.05)

        elif metric in ["consecutive_valid_steps", "max_consecutive_valid_steps"]:
            ax.set_ylim(-1, cfg.horizon + 1)

        # Runtime is usually skewed, so log scale is more readable.
        if metric == "runtime_sec":
            positive_values = raw_plot[metric][raw_plot[metric] > 0]

            if len(positive_values) > 0:
                ax.set_yscale("log")
                ax.set_ylabel("runtime_sec, log scale")
        # Real legend with the actual marker/line styles.
        legend_handles = [
            Line2D(
                [0],
                [0],
                marker="^",
                linestyle="None",
                markerfacecolor=mean_color,
                markeredgecolor=mean_color,
                markersize=7,
                label="Mean",
            ),
            Line2D(
                [0],
                [0],
                color=median_color,
                linewidth=1.8,
                label="Median",
            ),
        ]

        ax.legend(
            handles=legend_handles,
            loc="upper right",
            fontsize=9,
            frameon=True,
            framealpha=0.85,
            edgecolor="gray",
        )

        fig.tight_layout()

        out_path = plots_dir / f"boxplot_{metric}.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved: {out_path}")

In [ ]:
# -----------------------------
# Per-horizon-step validity ratio by method
# Counterfactual methods only
# -----------------------------

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


def load_raw_counterfactual_results_for_horizon(results_dir: Path) -> pd.DataFrame:
    """
    Load raw per-sample counterfactual results.

    This deliberately rebuilds from datasets/*/counterfactual_results.csv when needed,
    because all_counterfactual_results_raw.csv may be an older file without valid_h_XX columns.
    """
    result_files = list((results_dir / "datasets").glob("*/counterfactual_results.csv"))

    if not result_files:
        raw_path = results_dir / "all_counterfactual_results_raw.csv"
        if raw_path.exists():
            return pd.read_csv(raw_path)
        raise FileNotFoundError(
            f"No counterfactual_results.csv files found under {results_dir / 'datasets'}"
        )

    raw = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
    raw.to_csv(results_dir / "all_counterfactual_results_raw.csv", index=False)
    return raw


def plot_per_horizon_validity_ratio_methods_only(results_dir: Path) -> pd.DataFrame:
    """
    Plot exact per-horizon-step validity ratio.

    For each method and horizon step h:
        validity_ratio_at_step_h =
            fraction of successful counterfactual examples whose prediction at horizon h
            is inside the lower/upper bound at horizon h.

    This is not prefix/consecutive validity.
    It is computed directly from valid_h_01, ..., valid_h_24 columns.
    """
    raw = load_raw_counterfactual_results_for_horizon(results_dir)

    if "status" in raw.columns:
        raw = raw[raw["status"].astype(str) == "ok"].copy()

    valid_cols = sorted(
        [c for c in raw.columns if re.fullmatch(r"valid_h_\d+", c)],
        key=lambda c: int(c.split("_")[-1])
    )

    if not valid_cols:
        raise ValueError(
            "No valid_h_XX columns found in the saved results. "
            "Run the benchmark once after adding per_step_validity_dict(...) to run_dataset_benchmark. "
            "Use a fresh cfg.results_root or set cfg.resume_existing_results = False so old rows are not skipped."
        )

    plots_dir = results_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    horizon_steps = np.arange(1, len(valid_cols) + 1)
    rows = []

    for method in METHOD_ORDER:
        method_rows = raw[raw["method"] == method]

        if len(method_rows) == 0:
            continue

        step_values = method_rows[valid_cols].apply(pd.to_numeric, errors="coerce")

        # Mean of binary 0/1 validity columns = validity ratio at each horizon step.
        per_step_ratio = step_values.mean(axis=0).to_numpy(dtype=float)

        for h, value in zip(horizon_steps, per_step_ratio):
            rows.append({
                "method": method,
                "horizon_step": int(h),
                "validity_ratio_at_step": float(value),
                "n_examples": int(len(method_rows)),
            })

    horizon_df = pd.DataFrame(rows)

    out_csv = results_dir / "per_horizon_validity_ratio_methods_only.csv"
    horizon_df.to_csv(out_csv, index=False)

    plt.figure(figsize=(11, 6))

    for method in METHOD_ORDER:
        method_df = horizon_df[horizon_df["method"] == method]

        if len(method_df) == 0:
            continue

        plt.plot(
            method_df["horizon_step"],
            method_df["validity_ratio_at_step"],
            marker="o",
            linewidth=2,
            label=method,
        )

    plt.xlabel("Forecast horizon step")
    plt.ylabel("Validity ratio at horizon step")
    plt.title("Per-horizon-step validity ratio by method")
    plt.ylim(-0.05, 1.05)
    plt.xticks(horizon_steps)
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()

    out_plot = plots_dir / "per_horizon_validity_ratio_methods_only.png"
    plt.savefig(out_plot, dpi=220, bbox_inches="tight")
    plt.close()

    print(f"Saved plot: {out_plot}")
    print(f"Saved CSV:  {out_csv}")

    return horizon_df

In [ ]:
# -----------------------------
# Critical difference diagrams for all evaluation metrics
# Plus one overall average-rank diagram
# -----------------------------

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


def load_raw_counterfactual_results_for_cd(results_dir: Path) -> pd.DataFrame:
    """
    Load raw per-sample counterfactual results.

    Prefer rebuilding from datasets/*/counterfactual_results.csv because
    all_counterfactual_results_raw.csv can sometimes be older.
    """
    result_files = list((results_dir / "datasets").glob("*/counterfactual_results.csv"))

    if result_files:
        raw = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
        raw.to_csv(results_dir / "all_counterfactual_results_raw.csv", index=False)
        return raw

    raw_path = results_dir / "all_counterfactual_results_raw.csv"

    if raw_path.exists():
        return pd.read_csv(raw_path)

    raise FileNotFoundError(
        f"No counterfactual_results.csv files found under {results_dir / 'datasets'} "
        f"and no {raw_path} found."
    )


def nemenyi_critical_difference(k: int, n_blocks: int, alpha: float = 0.05) -> float:
    """
    Critical difference for the Nemenyi post-hoc test.

    k = number of methods
    n_blocks = number of paired benchmark cases/windows
    """
    try:
        from scipy.stats import studentized_range
        q_alpha = studentized_range.ppf(1 - alpha, k, np.inf) / math.sqrt(2)
    except Exception:
        # Common Nemenyi q_alpha values for alpha=0.05.
        # Enough for our current 7 methods, with fallback for nearby values.
        q_alpha_table_005 = {
            2: 1.960,
            3: 2.343,
            4: 2.569,
            5: 2.728,
            6: 2.850,
            7: 2.949,
            8: 3.031,
            9: 3.102,
            10: 3.164,
        }

        if alpha != 0.05:
            raise ValueError(
                "scipy is needed for alpha values other than 0.05."
            )

        if k not in q_alpha_table_005:
            raise ValueError(
                f"No fallback q_alpha value for k={k}. Install scipy or add it manually."
            )

        q_alpha = q_alpha_table_005[k]

    cd = q_alpha * math.sqrt(k * (k + 1) / (6.0 * n_blocks))
    return float(cd)


def prepare_metric_rank_table(raw: pd.DataFrame,
                              metric: str,
                              methods: list,
                              lower_is_better: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build paired benchmark table and corresponding ranks.

    Rows = same evaluated case/window.
    Columns = methods.
    Values = metric values.

    Returns:
        value_table: raw metric values per case and method
        rank_table: rank per case and method, where rank 1 is best
    """
    id_cols = [c for c in ["dataset", "test_idx", "start_idx"] if c in raw.columns]

    if not id_cols:
        raise ValueError(
            "Could not find dataset/test_idx/start_idx columns to define paired benchmark cases."
        )

    if metric not in raw.columns:
        raise ValueError(f"Metric column '{metric}' not found.")

    df = raw.copy()
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df = df.dropna(subset=[metric])

    # Only keep methods that exist.
    df = df[df["method"].isin(methods)].copy()

    # If duplicates exist, average them safely.
    value_table = (
        df.groupby(id_cols + ["method"], as_index=False)[metric]
        .mean()
        .pivot_table(index=id_cols, columns="method", values=metric, aggfunc="mean")
    )

    # Keep only cases where all methods are present.
    value_table = value_table.reindex(columns=methods).dropna(axis=0, how="any")

    if len(value_table) == 0:
        raise ValueError(
            f"No complete paired cases found for metric '{metric}'. "
            "A CD diagram needs the same cases evaluated by all methods."
        )

    rank_table = value_table.rank(
        axis=1,
        method="average",
        ascending=lower_is_better
    )

    return value_table, rank_table


def find_cd_cliques(avg_ranks_sorted: pd.Series, cd: float) -> list[tuple[int, int]]:
    """
    Find contiguous groups of sorted methods whose average-rank spread is <= CD.

    These are drawn as horizontal bars above the CD diagram.
    """
    ranks = avg_ranks_sorted.values
    k = len(ranks)
    groups = []

    for i in range(k):
        best_j = i
        for j in range(k - 1, i, -1):
            if ranks[j] - ranks[i] <= cd:
                best_j = j
                break

        if best_j > i:
            groups.append((i, best_j))

    # Remove groups that are strict subsets of another group.
    maximal = []
    for g in groups:
        i, j = g
        is_subset = False
        for h in groups:
            a, b = h
            if h != g and a <= i and j <= b:
                is_subset = True
                break
        if not is_subset:
            maximal.append(g)

    return maximal


def plot_critical_difference_diagram(avg_ranks: pd.Series,
                                     cd: float,
                                     title: str,
                                     out_path: Path,
                                     alpha: float = 0.05) -> None:
    """
    Draw a critical difference diagram similar to the common classifier-comparison CD plot.
    Lower rank is better.
    """
    avg_ranks = avg_ranks.sort_values()
    methods_sorted = list(avg_ranks.index)
    ranks_sorted = avg_ranks.values
    k = len(methods_sorted)

    fig_width = max(10, 1.15 * k + 4)
    fig, ax = plt.subplots(figsize=(fig_width, 4.8))

    ax.set_xlim(0.75, k + 0.25)
    ax.set_ylim(-1.65, 1.55)
    ax.axis("off")

    y_axis = 0.55

    # Main rank axis.
    ax.hlines(y_axis, 1, k, color="black", linewidth=1.4)

    for r in range(1, k + 1):
        ax.vlines(r, y_axis - 0.06, y_axis + 0.06, color="black", linewidth=1.2)
        ax.text(r, y_axis + 0.16, str(r), ha="center", va="bottom", fontsize=11)

    # Minor half ticks.
    for r in np.arange(1.5, k, 1.0):
        ax.vlines(r, y_axis - 0.035, y_axis + 0.035, color="black", linewidth=1.0)

    # CD bar.
    cd_start = 1.0
    cd_end = min(k, 1.0 + cd)
    y_cd = 1.15
    ax.hlines(y_cd, cd_start, cd_end, color="black", linewidth=1.4)
    ax.vlines([cd_start, cd_end], y_cd - 0.08, y_cd + 0.08, color="black", linewidth=1.4)
    ax.text((cd_start + cd_end) / 2, y_cd + 0.13, "CD", ha="center", va="bottom", fontsize=12)

    # Method labels and connector lines.
    left_methods = []
    right_methods = []

    for idx, method in enumerate(methods_sorted):
        if idx < math.ceil(k / 2):
            left_methods.append((idx, method, ranks_sorted[idx]))
        else:
            right_methods.append((idx, method, ranks_sorted[idx]))

    left_y_positions = np.linspace(0.15, -1.10, len(left_methods))
    right_y_positions = np.linspace(0.15, -1.10, len(right_methods))

    for (_, method, rank), y in zip(left_methods, left_y_positions):
        ax.vlines(rank, y_axis, y, color="black", linewidth=1.0)
        ax.hlines(y, 0.75, rank, color="black", linewidth=1.0)
        ax.text(0.70, y, method, ha="right", va="center", fontsize=10)

    for (_, method, rank), y in zip(right_methods, right_y_positions):
        ax.vlines(rank, y_axis, y, color="black", linewidth=1.0)
        ax.hlines(y, rank, k + 0.25, color="black", linewidth=1.0)
        ax.text(k + 0.30, y, method, ha="left", va="center", fontsize=10)

    # Non-significant cliques.
    cliques = find_cd_cliques(avg_ranks, cd)
    y_bar = 0.82

    for group_i, group_j in cliques:
        x1 = ranks_sorted[group_i]
        x2 = ranks_sorted[group_j]
        ax.hlines(y_bar, x1, x2, color="black", linewidth=4.0)
        y_bar -= 0.13

    ax.text(
        (1 + k) / 2,
        1.45,
        title,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
    )

    ax.text(
        (1 + k) / 2,
        -1.50,
        f"Lower rank is better. Horizontal thick bars connect methods not significantly different by Nemenyi CD, α={alpha}.",
        ha="center",
        va="center",
        fontsize=9,
    )

    fig.tight_layout()
    fig.savefig(out_path, dpi=250, bbox_inches="tight")
    plt.close(fig)


def make_critical_difference_diagrams(results_dir: Path,
                                      metrics: list[str] | None = None,
                                      metric_directions: dict[str, str] | None = None,
                                      alpha: float = 0.05,
                                      make_overall_rank_diagram: bool = True) -> pd.DataFrame:
    """
    Create one critical difference diagram per metric, plus an optional overall-rank CD diagram.

    Direction convention:
        "higher" means higher metric value is better.
        "lower" means lower metric value is better.

    Output:
        plots/critical_diagram_<metric>.png
        plots/critical_diagram_overall_average_rank.png
        critical_diagram_average_ranks.csv
    """
    results_dir = Path(results_dir)
    plots_dir = results_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    raw = load_raw_counterfactual_results_for_cd(results_dir)

    if "status" in raw.columns:
        print("Status counts:")
        print(raw["status"].value_counts(dropna=False))
        raw = raw[raw["status"].astype(str) == "ok"].copy()

    methods = [m for m in METHOD_ORDER if m in raw["method"].unique()]

    if len(methods) < 2:
        raise ValueError("Need at least two methods for a critical difference diagram.")

    default_metric_directions = {
        # Higher is better
        "validity_ratio": "higher",
        "stepwise_validity_auc": "higher",
        "consecutive_valid_steps": "higher",
        "max_consecutive_valid_steps": "higher",

        # Lower is better
        "distance_to_band": "lower",
        "proximity_l2_scaled": "lower",
        "proximity_l1_scaled": "lower",
        "sparsity_count": "lower",
        "sparsity_ratio": "lower",
        "runtime_sec": "lower",
    }

    if metric_directions is None:
        metric_directions = default_metric_directions
    else:
        temp = default_metric_directions.copy()
        temp.update(metric_directions)
        metric_directions = temp

    if metrics is None:
        metrics = [
            m for m in metric_directions.keys()
            if m in raw.columns
        ]

    all_summary_rows = []
    rank_tables_for_overall = []

    for metric in metrics:
        if metric not in raw.columns:
            print(f"Skipping {metric}: column not found.")
            continue

        direction = metric_directions.get(metric)

        if direction not in ["higher", "lower"]:
            raise ValueError(
                f"Direction for metric '{metric}' must be either 'higher' or 'lower'."
            )

        lower_is_better = direction == "lower"

        try:
            value_table, rank_table = prepare_metric_rank_table(
                raw=raw,
                metric=metric,
                methods=methods,
                lower_is_better=lower_is_better,
            )
        except ValueError as e:
            print(f"Skipping {metric}: {e}")
            continue

        n_blocks = len(rank_table)
        k = len(methods)
        cd = nemenyi_critical_difference(k=k, n_blocks=n_blocks, alpha=alpha)
        avg_ranks = rank_table.mean(axis=0).sort_values()

        for method in avg_ranks.index:
            all_summary_rows.append({
                "diagram": metric,
                "metric": metric,
                "method": method,
                "average_rank": float(avg_ranks[method]),
                "n_blocks": int(n_blocks),
                "n_methods": int(k),
                "critical_difference": float(cd),
                "direction": direction,
            })

        out_path = plots_dir / f"critical_diagram_{metric}.png"
        title = f"Critical difference diagram: {metric}"
        plot_critical_difference_diagram(
            avg_ranks=avg_ranks,
            cd=cd,
            title=title,
            out_path=out_path,
            alpha=alpha,
        )

        print(f"Saved: {out_path}")

        # For overall rank diagram:
        # each metric contributes one rank table.
        # We average metric ranks per case and method.
        rank_tables_for_overall.append(rank_table.copy())

    if make_overall_rank_diagram and rank_tables_for_overall:
        # Align all rank tables on common cases.
        common_index = rank_tables_for_overall[0].index

        for rt in rank_tables_for_overall[1:]:
            common_index = common_index.intersection(rt.index)

        if len(common_index) == 0:
            print("Skipping overall rank diagram: no common paired cases across metrics.")
        else:
            aligned = [
                rt.loc[common_index, methods]
                for rt in rank_tables_for_overall
            ]

            # Average rank across metrics for each case and method.
            overall_score_table = sum(aligned) / len(aligned)

            # Then rank methods by this overall average rank per case.
            # Lower overall score is better.
            overall_rank_table = overall_score_table.rank(
                axis=1,
                method="average",
                ascending=True,
            )

            n_blocks = len(overall_rank_table)
            k = len(methods)
            cd = nemenyi_critical_difference(k=k, n_blocks=n_blocks, alpha=alpha)
            avg_ranks = overall_rank_table.mean(axis=0).sort_values()

            for method in avg_ranks.index:
                all_summary_rows.append({
                    "diagram": "overall_average_rank",
                    "metric": "overall_average_rank",
                    "method": method,
                    "average_rank": float(avg_ranks[method]),
                    "n_blocks": int(n_blocks),
                    "n_methods": int(k),
                    "critical_difference": float(cd),
                    "direction": "lower",
                })

            out_path = plots_dir / "critical_diagram_overall_average_rank.png"
            plot_critical_difference_diagram(
                avg_ranks=avg_ranks,
                cd=cd,
                title="Critical difference diagram: overall average rank",
                out_path=out_path,
                alpha=alpha,
            )

            print(f"Saved: {out_path}")

    summary_df = pd.DataFrame(all_summary_rows)
    summary_path = results_dir / "critical_diagram_average_ranks.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"Saved: {summary_path}")

    return summary_df

## Verify exact model, 50/10/40 chronological split, training scaler, feature order, and configurable intervention subset before a run

In [ ]:
# Inspection only: verify CSV, saved metadata and model checkpoint paths for a dataset.
# EXAMPLE: inspect_public_dataset('Weather')   # optional, no benchmark run

def inspect_public_dataset(name):
    p=configure_public_dataset(name)
    bundle=prepare_dataset(p)
    model=load_best_gru(bundle)
    y0=predict_model(model,bundle.test_ds[0][0].numpy())
    print('Forecaster test:',bundle.name,'X',bundle.test_ds[0][0].shape,'yhat',y0.shape,
          'training features:',bundle.feature_cols,'editable:',cfg.change_feature_names)
    print('Checkpoint:',find_saved_model_path(name))
    return {'dataset':name,'test_windows':len(bundle.test_ds),'input_len':cfg.input_len,
            'horizon':cfg.horizon,'editable':cfg.change_feature_names}

## Per-dataset execution orchestrator; preserves the original seven methods, metrics and plotting functions

In [ ]:
# ============= Shared orchestrator for four independent dataset run cells =============
import json
from pathlib import Path

def run_one_public_dataset(name: str, include_all_plots: bool = True) -> Path:
    path = configure_public_dataset(name)
    results_dir = safe_mkdir(cfg.results_root)
    safe_mkdir(results_dir/'datasets');safe_mkdir(results_dir/'plots')
    dataset_dir=safe_mkdir(results_dir/'datasets'/name)
    manifest_path=dataset_dir/'dataset_manifest.json'
    manifest={'dataset':name,'csv':str(path),'csv_sha256':sha256_file(path),
              'checkpoint':str(find_saved_model_path(name)),
              'checkpoint_sha256':sha256_file(find_saved_model_path(name)),
              'input_len':cfg.input_len,'horizon':cfg.horizon,'target':cfg.target_col,
              'editable_exogenous':list(cfg.change_feature_names),'input_feature_cols':ACTIVE_MODEL_METADATA['feature_cols'],
              'n_test_windows_limit':cfg.max_test_windows_per_dataset,
              'experiment_config':{k:str(v) if isinstance(v,Path) else v for k,v in asdict(cfg).items()}}
    if manifest_path.exists():
        old=json.loads(manifest_path.read_text())
        # prevents mixing old/new test window selection, target, CSV, or different model weights
        critical=('csv_sha256','checkpoint_sha256','input_len','horizon','target','editable_exogenous','n_test_windows_limit')
        if any(old.get(k)!=manifest[k] for k in critical):
            raise ValueError(f'Existing {name} results were generated with a different dataset/model/settings; use a NEW RESULTS_ROOT_PUBLIC or explicitly archive the old output.')
    manifest_path.write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
    # Full-test forecast re-check ensures identical model and preprocessing, no CF work before validation.
    bundle=prepare_dataset(path)
    model=load_best_gru(bundle)
    check=evaluate_forecaster(model,bundle)
    scores_path=cfg.previous_output_root/'forecast_metrics.csv'
    if scores_path.exists():
        trained=pd.read_csv(scores_path)
        r=trained.loc[trained.dataset.astype(str)==name]
        if len(r)==1:
            ref=r.iloc[0]
            print('Saved full-test MAE:',ref.test_mae_full,'| CF reload full-test MAE:',check['test_mae'])
            if not np.isclose(check['test_mae'],float(ref.test_mae_full),rtol=1e-3,atol=1e-3):
                raise ValueError('FULL test MAE mismatches saved training run; inspect checkpoint and data before generating CFs')
            if not np.isclose(check['test_r2'],float(ref.test_r2_full),rtol=1e-3,atol=1e-3):
                raise ValueError('FULL test R² mismatches saved training run')
    del model, bundle
    print(f'Running ALL seven original CF variants on {name}; output: {dataset_dir}')
    cf_df,forecast_metrics=run_dataset_benchmark(path,results_dir)
    # refresh overall forecast metrics for ALL completed public datasets, not just latest one
    forecasts=[]
    for mp in sorted((results_dir/'datasets').glob('*/forecast_metrics.json')):
        forecasts.append(json.loads(mp.read_text()))
    if forecasts:pd.DataFrame(forecasts).to_csv(results_dir/'forecast_recheck_metrics.csv',index=False)
    (results_dir/'config.json').write_text(json.dumps({k:str(v) if isinstance(v,Path) else v for k,v in asdict(cfg).items()},indent=2),encoding='utf-8')
    raw,overall=aggregate_and_plot(results_dir)
    write_method_report(results_dir)
    if include_all_plots:
        make_metric_boxplots(results_dir)
        horizon_df=plot_per_horizon_validity_ratio_methods_only(results_dir)
        print('Saved per-step validity:',results_dir/'per_horizon_validity_ratio_methods_only.csv')
        try:
            rank_df=make_critical_difference_diagrams(
                results_dir,
                metrics=['validity_ratio','stepwise_validity_auc','consecutive_valid_steps',
                         'max_consecutive_valid_steps','distance_to_band','proximity_l2_scaled',
                         'proximity_l1_scaled','sparsity_count','sparsity_ratio','runtime_sec'],
                alpha=0.05,make_overall_rank_diagram=True)
            print('Saved critical-difference summary:',results_dir/'critical_diagram_average_ranks.csv')
        except ValueError as e:
            print('Critical-difference plot unavailable until complete paired rows exist:',e)
    print('DONE:',name,'| CSV:',dataset_dir/'counterfactual_results.csv','| combined outputs:',results_dir)
    return dataset_dir

## RUN ONLY ETTh1 (independent section)

Execute this single cell when you want the complete seven-method CF benchmark for **ETTh1**. It creates/updates only `EDBT-results-public-datasets/datasets/ETTh1` and refreshes cross-public summary figures from any datasets already completed. For a laptop trial, first set `cfg.max_test_windows_per_dataset=2` **and change** `RESULTS_ROOT_PUBLIC`/`cfg.results_root` to a different pilot folder; restore the original 200 for the real run.

In [ ]:
# ETTh1 — FULL benchmark, all plots/tables/CSV/report and checkpoint verification
results_etth1 = run_one_public_dataset('ETTh1')
results_etth1

## RUN ONLY ETTh2 (independent section)

Execute this single cell when you want the complete seven-method CF benchmark for **ETTh2**. It creates/updates only `EDBT-results-public-datasets/datasets/ETTh2` and refreshes cross-public summary figures from any datasets already completed. For a laptop trial, first set `cfg.max_test_windows_per_dataset=2` **and change** `RESULTS_ROOT_PUBLIC`/`cfg.results_root` to a different pilot folder; restore the original 200 for the real run.

In [ ]:
# ETTh2 — FULL benchmark, all plots/tables/CSV/report and checkpoint verification
results_etth2 = run_one_public_dataset('ETTh2')
results_etth2

## RUN ONLY Weather (independent section)

Execute this single cell when you want the complete seven-method CF benchmark for **Weather**. It creates/updates only `EDBT-results-public-datasets/datasets/Weather` and refreshes cross-public summary figures from any datasets already completed. For a laptop trial, first set `cfg.max_test_windows_per_dataset=2` **and change** `RESULTS_ROOT_PUBLIC`/`cfg.results_root` to a different pilot folder; restore the original 200 for the real run.

In [ ]:
# Weather — FULL benchmark, all plots/tables/CSV/report and checkpoint verification
results_weather = run_one_public_dataset('Weather')
results_weather

## RUN ONLY Electricity (independent section)

Execute this single cell when you want the complete seven-method CF benchmark for **Electricity**. It creates/updates only `EDBT-results-public-datasets/datasets/Electricity` and refreshes cross-public summary figures from any datasets already completed. For a laptop trial, first set `cfg.max_test_windows_per_dataset=2` **and change** `RESULTS_ROOT_PUBLIC`/`cfg.results_root` to a different pilot folder; restore the original 200 for the real run.

In [ ]:
# Electricity — FULL benchmark, all plots/tables/CSV/report and checkpoint verification
results_electricity = run_one_public_dataset('Electricity')
results_electricity

## Outputs / experiment interpretation

Each dataset saves its original `counterfactual_results.csv`, `forecast_metrics.json`, `ga_granger_screen.csv`, `cf_example_plots/` and a new `dataset_manifest.json`. Combined outputs include `all_counterfactual_results_raw.csv`, original mean/STD/min/max/count tables, all bar/error-bar/tradeoff/rank plots, boxplots, per-step validity ratios, CD diagrams, and `method_and_experiment_report.md`. The aggregate figures reflect **only datasets you have run so far**. An individual dataset's CF methods all use the same frozen GRU and three preselected exogenous channels.

**Limitations carried over unchanged:** GA screens selected channels with conventional Granger; DiffusionCF uses the notebook's simplified VAE/DDPM adaptation; COMET's historic penalty/activity rule and original bounds are not domain-validated on electricity or meteorology; a weather temperature target is not naturally “actionable.” Runtime scales with 321 forecaster inputs on Electricity. 